# NB_LIB_EXTRACT_RAWDATA — Thư viện hàm dùng chung cho các notebook extract raw → silver

**Cách dùng:** trong notebook xử lý, đặt một cell riêng ngay sau cell tham số:
```
%run NB_LIB_EXTRACT_RAWDATA
```
`%run` chạy notebook này trong cùng context, nên mọi hằng số, class, hàm bên dưới dùng được trực tiếp.

**Quy ước của notebook này**
- Chỉ định nghĩa hằng số, class, hàm. **Không có luồng main**, không đọc/ghi dữ liệu khi được `%run`, không `spark.conf.set`.
- Không phụ thuộc nguồn cụ thể (partner, poi, cms…): mọi thứ riêng của nguồn truyền qua tham số hoặc đọc từ bảng control.
- Đặt cùng default lakehouse với notebook gọi (`lh_vv_bronze`); tên bảng luôn viết đầy đủ `lakehouse.schema.table`.
- Bảng control ở `lh_vv_ctrl.dbo` (`CTRL_SCHEMA`).
- Mỗi hàm có docstring: mục đích, Input, Output, ví dụ.

| # | Nhóm | Nội dung |
|---|---|---|
| 1 | `constants` | Tên bảng control, cột kỹ thuật bảng silver, mẫu tên hợp lệ |
| 2 | `common_utils` | Thời gian UTC, chuẩn hoá tham số, rút gọn lỗi |
| 3 | `log_utils` | Ghi log console an toàn khi chạy nhiều luồng |
| 4 | `validation_utils` | Kiểm tra tên bảng/cột, quote tên, chuẩn hoá kiểu dữ liệu |
| 5 | `retry_utils` | Nhận diện lỗi tạm thời, thử lại khi ghi Delta xung đột |
| 6 | `convert_utils` | Chuyển chuỗi JSON sang kiểu đích theo `convert_rule` |
| 7 | `json_utils` | Parse chuỗi JSON thành struct, phát hiện JSON hỏng |
| 8 | `data_models` | Dataclass cấu hình và kết quả: `ColumnSpec`, `TableSpec`, `RunContext`, `Profile`, `TableResult` |
| 9 | `config_utils` | Nạp + kiểm tra cấu hình từ `ctrl_mng_pipeline_config` và `ctrl_cfg_schema_registry` |
| 10 | `delta_utils` | Version bảng, đọc file mới theo `_delta_log`, tạo/bổ sung cột bảng đích, MERGE trả số dòng, append theo schema |
| 11 | `run_control_utils` | Watermark, chốt snapshot, đọc raw tăng dần (VERSION / FULL), khoá chống chạy chồng, run log, table log, trạng thái run |
| 12 | `parallel_utils` | Chạy song song theo wave priority |
| 13 | `state_reject_utils` | **[03/10 3P]** State theo entity (`ctrl_cdc_state`, chống lùi), MERGE `ctrl_cdc_reject` — chuyển từ notebook partner sang |
| 14 | `table_flow_utils` | **[03/10 3P]** Khung chạy dùng chung: 1 bảng có retry, NO_DATA + wave, cả lần chạy (khoá → run log → xử lý → watermark → nhả khoá) |

**Thay đổi 03/10 (thêm nguồn 3rd-party `brz_3rd_crawler_poi_stream`)** — đánh dấu `[03/10 3P]` trong code

| Chỗ | Thay đổi | Ảnh hưởng `NB_EXTRACT_PARTNER_CDC_BRZ_TO_SLV` |
|---|---|---|
| §1 | Thêm `LOAD_MODES`, `SNAPSHOT_MODES`, `SNAPSHOT_TECH_COLUMNS`, `PATH_SCOPES` | Không |
| §6 | Thêm `convert_rule` `LOWER_TRIM` (chuẩn hoá khoá chuỗi) và 3 rule khoá suy ra `HASH_MD5_UUID`, `HASH_SHA256`, `HASH_SHA256_PIPE` (`derived_sql`) | Không (partner không dùng) |
| §8 | `ColumnSpec`: `paths`, `is_derived`; `TableSpec`: thêm `load_mode`, `align_path`, `dedup_order` (có mặc định), `is_snapshot`, `tech_columns`; thêm `RunOutcome` | Không: giá trị mặc định = hành vi cũ |
| §9 | `build_table_specs` kiểm tra cấu hình theo `load_mode`; `load_table_specs(ctx, allowed_modes=("CDC",))` đọc thêm 3 cột mới của `ctrl_mng_pipeline_config` (chưa có cột thì coi NULL) | Partner gọi như cũ; dòng partner có `load_mode` khác `CDC` → `ConfigError` |
| §10 | `target_ddl` / `ensure_target`: cột kỹ thuật mặc định theo `spec.tech_columns` | Không: spec CDC vẫn `TECH_COLUMNS` |
| §5 | `with_retry`: 4 → 6 lần, thêm jitter (2 nguồn có thể ghi bảng control cùng lúc) | Chỉ thêm lượt thử lại khi xung đột ghi |
| §13, §14 | Hàm chuyển từ partner (logic giữ nguyên) + khung chạy dùng chung; `commit_state` / `merge_rejects` nhận thêm `src_tbl` (điều kiện partition) | Partner **xoá** bản riêng và gọi bản ở lib — xem ghi chú `[03/10 → LIB]` trong partner. **Deploy lib trước partner** |


## 1. constants — Hằng số dùng chung

| Hằng số | Ý nghĩa |
|---|---|
| `T_PIPELINE_CONFIG`, `T_WATERMARK`, `T_REGISTRY`, `T_LOG_RUN`, `T_LOG_TABLE_RUN`, `T_CDC_STATE`, `T_CDC_REJECT` | Tên đầy đủ 7 bảng control trong `lh_vv_ctrl.dbo` |
| `TECH_COLUMNS` | Cột kỹ thuật của bảng silver: `deleted`, `_ingested_at`, `_source_db` (giữ như bảng `partner.*` cũ) |
| `IDENT_RE` | Mẫu tên schema/bảng/cột hợp lệ: chữ cái đầu, sau đó chữ, số, `_`. Không cho bắt đầu bằng `_` vì tiền tố này dành cho cột nội bộ của notebook |
| `JSON_PATH_RE` | Mẫu đường dẫn JSON hợp lệ: `id`, `address.city` |
| `MIN_TS` | Mốc 1900-01-01, dùng thay NULL khi so sánh thứ tự thời gian |
| `LOAD_MODES`, `SNAPSHOT_MODES` | **[03/10 3P]** Cách tách dòng của bảng đích (`ctrl_mng_pipeline_config.load_mode`), xem bảng dưới |
| `SNAPSHOT_TECH_COLUMNS` | **[03/10 3P]** Cột kỹ thuật bảng đích của nguồn snapshot: `_crawled_at`, `_event_id`, `_first_seen_at`, `_last_seen_at`, `_ingested_at` |
| `PATH_SCOPES` | **[03/10 3P]** Tiền tố `json_path` đặc biệt của bảng snapshot, xem bảng dưới |

**`load_mode`** (NULL = `CDC`):

| load_mode | 1 dòng đích = | `src_object` (registry) | Ví dụ |
|---|---|---|---|
| `CDC` | 1 event Debezium của bảng `src_object` | Bảng Postgres (`source.table`) | partner |
| `DOC` | 1 tài liệu (event) | Khối object trong payload; `$` = gốc payload | `slv_3p_poi` (`$`), `slv_3p_poi_address` (`poi_address`) |
| `DOC_ARRAY` | 1 phần tử mảng trong tài liệu | Mảng trong payload | `slv_3p_poi_review` (`poi_review`) |
| `LANG` | 1 object bản địa hoá (theo ngôn ngữ) | Đường dẫn trong object bản địa hoá; `$` = cả object | `slv_3p_poi_enrichment` (`$`) |
| `LANG_ARRAY` | 1 phần tử mảng trong object bản địa hoá | Mảng trong object bản địa hoá | `slv_3p_poi_review_i18n` (`reviews`) |

**`json_path` của bảng snapshot** (bảng CDC giữ như cũ: đường dẫn trong `before`/`after`):

| json_path | Lấy gì |
|---|---|
| `a.b` | Trường trong phần tử / khối (DOC `$`: trong gốc payload) |
| `$` | Cả phần tử / khối dạng chuỗi JSON (không dùng với DOC `$`) |
| `_doc.<cột>` | Cột của bảng raw, vd `_doc.source_name` |
| `_root.a.b` | Trường tính từ gốc payload |
| `_langs` | Mảng JSON các ngôn ngữ có trong tài liệu (đã `lower(trim)`, chưa lọc theo tham số `langs`) |
| `_lang` | Ngôn ngữ của dòng (`LANG`, `LANG_ARRAY`) |
| `_pos` | Vị trí phần tử trong mảng, từ 0 (`DOC_ARRAY`, `LANG_ARRAY`) |
| `_align.a` | Trường của phần tử cùng vị trí trong mảng `align_path` (chỉ khi 2 mảng cùng số phần tử) |
| `p1,p2,...` | Danh sách đường dẫn, chỉ dùng với rule khoá suy ra `HASH_*` |


In [ ]:
# =============================================================================
# 1. constants
# =============================================================================
import json
import random                                            # [03/10 3P] jitter cho with_retry
import re
import threading
import time
import traceback                                         # [03/10 3P] run_controlled
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
from functools import reduce                             # [03/10 3P] commit_state, merge_rejects
from typing import Callable, Optional
from urllib.parse import unquote

from delta.tables import DeltaTable
from pyspark import StorageLevel, inheritable_thread_target
from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (ArrayType, BooleanType, IntegerType, LongType, MapType, StringType, StructField,
                               StructType)

try:
    from pyspark.errors import AnalysisException          # Spark >= 3.4
except ImportError:                                        # pragma: no cover
    from pyspark.sql.utils import AnalysisException

CTRL_SCHEMA       = "lh_vv_ctrl.dbo"
T_PIPELINE_CONFIG = f"{CTRL_SCHEMA}.ctrl_mng_pipeline_config"
T_WATERMARK       = f"{CTRL_SCHEMA}.ctrl_mng_watermark"
T_REGISTRY        = f"{CTRL_SCHEMA}.ctrl_cfg_schema_registry"
T_LOG_RUN         = f"{CTRL_SCHEMA}.ctrl_log_run"
T_LOG_TABLE_RUN   = f"{CTRL_SCHEMA}.ctrl_log_table_run"
T_CDC_STATE       = f"{CTRL_SCHEMA}.ctrl_cdc_state"
T_CDC_REJECT      = f"{CTRL_SCHEMA}.ctrl_cdc_reject"

TECH_COLUMNS = (("deleted", "boolean"), ("_ingested_at", "timestamp"), ("_source_db", "string"))

IDENT_RE     = re.compile(r"[A-Za-z][A-Za-z0-9_]*")
JSON_PATH_RE = re.compile(r"[A-Za-z0-9_]+(\.[A-Za-z0-9_]+)*")
MIN_TS       = datetime(1900, 1, 1)

# [03/10 3P] Nguồn snapshot (3rd-party): 1 event = 1 tài liệu đầy đủ của 1 địa điểm, không có xoá / TOAST
LOAD_MODES     = ("CDC", "DOC", "DOC_ARRAY", "LANG", "LANG_ARRAY")
SNAPSHOT_MODES = ("DOC", "DOC_ARRAY", "LANG", "LANG_ARRAY")
SNAPSHOT_TECH_COLUMNS = (("_crawled_at", "timestamp"), ("_event_id", "string"), ("_first_seen_at", "timestamp"),
                         ("_last_seen_at", "timestamp"), ("_ingested_at", "timestamp"))
PATH_SCOPES = ("_doc", "_root", "_align", "_lang", "_pos", "_langs")   # tiền tố json_path đặc biệt (bảng snapshot)


## 2. common_utils — Tiện ích chung

| Hàm | Mục đích |
|---|---|
| `utc_now()` | Thời điểm hiện tại theo UTC, không kèm múi giờ (để ghi cột TIMESTAMP) |
| `to_bool(value)` | Chuẩn hoá tham số bool; pipeline có thể truyền `"true"`, `"1"`… |
| `split_csv(value)` | Tách chuỗi danh sách `"a, b"` thành `["a", "b"]` (vd tham số `table_filter`) |
| `short_error(e)` | Rút gọn lỗi thành chuỗi ≤ 4000 ký tự để ghi cột `error_message` |


In [ ]:
# =============================================================================
# 2. common_utils
# =============================================================================
def utc_now() -> datetime:
    """Thời điểm hiện tại theo UTC, bỏ thông tin múi giờ.

    Input : không
    Output: datetime không kèm tzinfo, giá trị theo giờ UTC
    Ví dụ : lúc 16:30 giờ Việt Nam -> datetime(2026, 10, 2, 9, 30, ...)
    """
    return datetime.now(timezone.utc).replace(tzinfo=None)


def to_bool(value) -> bool:
    """Chuẩn hoá giá trị bool từ tham số notebook.

    Input : value — bool | str | int | None
    Output: True nếu value là "1", "true", "yes", "y" (không phân biệt hoa thường), ngược lại False
    Ví dụ : to_bool(True) -> True ; to_bool("TRUE") -> True ; to_bool("0") -> False ; to_bool(None) -> False
    """
    return str(value).strip().lower() in ("1", "true", "yes", "y")


def split_csv(value) -> list:
    """Tách chuỗi phân cách bởi dấu phẩy, bỏ khoảng trắng và phần tử rỗng.

    Input : value — str | None, vd " slv_products, slv_orders ,,"
    Output: list[str]
    Ví dụ : split_csv(" slv_products, slv_orders ,,") -> ["slv_products", "slv_orders"] ; split_csv("") -> []
    """
    return [x.strip() for x in str(value or "").split(",") if x.strip()]


def short_error(e: BaseException, limit: int = 4000) -> str:
    """Chuỗi lỗi ngắn gọn: "<TênLỗi>: <nội dung>", cắt còn tối đa `limit` ký tự.

    Input : e — exception ; limit — số ký tự tối đa (mặc định 4000)
    Output: str
    Ví dụ : short_error(ValueError("sai kiểu")) -> "ValueError: sai kiểu"
    """
    return f"{type(e).__name__}: {e}"[:limit]


## 3. log_utils — Ghi log

| Hàm | Mục đích |
|---|---|
| `log(msg, level, wave, table)` | In 1 dòng log có giờ UTC, mức, wave, bảng. Dùng khoá để các luồng song song không in chồng lên nhau |


In [ ]:
# =============================================================================
# 3. log_utils
# =============================================================================
_LOG_LOCK = threading.Lock()


def log(msg: str, level: str = "INFO", wave: Optional[int] = None, table: Optional[str] = None) -> None:
    """In 1 dòng log định dạng cố định, an toàn khi gọi từ nhiều luồng.

    Input : msg — nội dung ; level — INFO | WARN | ERROR ; wave — số wave (tuỳ chọn) ; table — tên bảng (tuỳ chọn)
    Output: không (in ra console)
    Ví dụ : log("ok in=12", wave=1, table="slv_products")
            -> [09:41:31][INFO][w1][slv_products] ok in=12
    """
    tags = f"[{datetime.now(timezone.utc):%H:%M:%S}][{level}]"
    tags += f"[w{wave}]" if wave is not None else ""
    tags += f"[{table}]" if table else ""
    with _LOG_LOCK:
        print(f"{tags} {msg}", flush=True)


## 4. validation_utils — Kiểm tra tên và kiểu dữ liệu

Tên bảng/cột lấy từ bảng cấu hình sẽ được ghép vào câu SQL, nên phải kiểm tra trước để tránh lỗi cú pháp và chèn SQL.

| Hàm | Mục đích |
|---|---|
| `is_identifier(name)` | Tên bảng/cột/schema có hợp lệ không |
| `is_fq_name(name)` | Tên đầy đủ nhiều phần (`lakehouse.schema.table`) có hợp lệ không |
| `quote_name(fq_name)` | Bọc từng phần tên bằng backtick để ghép vào SQL |
| `canonical_type(data_type)` | Nhờ Spark xác nhận kiểu dữ liệu và trả tên chuẩn để so với bảng đích |


In [ ]:
# =============================================================================
# 4. validation_utils
# =============================================================================
_TYPE_CACHE: dict = {}


def is_identifier(name) -> bool:
    """Tên 1 phần (schema, bảng, cột) có hợp lệ không.

    Input : name — str | None
    Output: bool. Hợp lệ khi bắt đầu bằng chữ cái, sau đó chỉ có chữ, số, "_"
    Ví dụ : is_identifier("slv_products") -> True ; is_identifier("_raw") -> False ; is_identifier("a-b") -> False
    """
    return bool(name) and IDENT_RE.fullmatch(name) is not None


def is_fq_name(name) -> bool:
    """Tên nhiều phần ngăn bởi dấu chấm có hợp lệ không (mọi phần đều là identifier).

    Input : name — str, vd "lh_vv_bronze.dbo.partner_raw_data"
    Output: bool
    Ví dụ : is_fq_name("lh_vv_silver.dbo") -> True ; is_fq_name("lh.dbo.`x`") -> False
    """
    return bool(name) and all(is_identifier(part) for part in name.split("."))


def quote_name(fq_name: str) -> str:
    """Bọc từng phần của tên bằng backtick để dùng trong Spark SQL.

    Input : fq_name — tên đã kiểm tra bằng is_fq_name, vd "lh_vv_silver.dbo.slv_x"
    Output: str
    Ví dụ : quote_name("lh_vv_silver.dbo.slv_x") -> "`lh_vv_silver`.`dbo`.`slv_x`"
    """
    return ".".join(f"`{part}`" for part in fq_name.split("."))


def canonical_type(data_type: str) -> str:
    """Kiểm tra kiểu dữ liệu bằng Spark (CAST(NULL AS <kiểu>)) và trả về tên chuẩn.
    Kết quả được cache theo tên kiểu.

    Input : data_type — str, vd "BIGINT", "decimal(19, 4)", "Timestamp"
    Output: str tên chuẩn của Spark (simpleString). Kiểu sai -> ValueError
    Ví dụ : canonical_type("BIGINT") -> "bigint" ; canonical_type("decimal(19, 4)") -> "decimal(19,4)"
            canonical_type("strng") -> ValueError("data_type không hợp lệ 'strng'")
    """
    key = re.sub(r"\s+", "", (data_type or "").lower())
    if key not in _TYPE_CACHE:
        try:
            if not re.fullmatch(r"[a-z]+(\(\d+(,\d+)?\))?", key):     # chặn ký tự lạ trước khi ghép SQL
                raise ValueError(key)
            _TYPE_CACHE[key] = spark.sql(f"SELECT CAST(NULL AS {key}) AS x").schema[0].dataType.simpleString()
        except Exception:
            raise ValueError(f"data_type không hợp lệ '{data_type}'") from None
    return _TYPE_CACHE[key]


## 5. retry_utils — Thử lại khi lỗi tạm thời

Bảng control dùng chung giữa nhiều luồng (partner, poi…), hai lần ghi cùng lúc vào một bảng Delta có thể xung đột.
Chỉ thử lại lỗi tạm thời; lỗi logic (sai cột, sai kiểu) không thử lại.

| Hàm / hằng số | Mục đích |
|---|---|
| `TRANSIENT_MARKERS` | Dấu hiệu lỗi tạm thời: xung đột ghi Delta, capacity bận (HTTP 430), mất kết nối |
| `is_transient(e)` | Lỗi có phải lỗi tạm thời không |
| `with_retry(fn, label)` | Chạy `fn`, lỗi tạm thời thì chờ tăng dần (2s, 4s, 8s, 16s, 32s + 0–1s ngẫu nhiên) rồi chạy lại. **[03/10 3P]** 4 → 6 lần + jitter: 2 nguồn có thể ghi bảng control cùng lúc |


In [ ]:
# =============================================================================
# 5. retry_utils
# =============================================================================
TRANSIENT_MARKERS = (
    "ConcurrentAppendException", "ConcurrentDeleteReadException", "ConcurrentDeleteDeleteException",
    "ConcurrentTransactionException", "ConcurrentWriteException", "MetadataChangedException",
    "ProtocolChangedException", "TooManyRequestsForCapacity", "Connection reset",
)


def is_transient(e: BaseException) -> bool:
    """Lỗi có phải lỗi tạm thời (nên thử lại) không. So theo tên lớp lỗi và nội dung.

    Input : e — exception (kể cả lỗi Java bọc qua Py4J)
    Output: bool
    Ví dụ : is_transient(Exception("io.delta.exceptions.ConcurrentAppendException: ...")) -> True
            is_transient(ValueError("cột không tồn tại")) -> False
    """
    text = f"{type(e).__name__} {e}"
    return any(marker in text for marker in TRANSIENT_MARKERS)


def with_retry(fn: Callable, label: str, attempts: int = 6, base_s: float = 2.0):
    """Chạy fn(); nếu lỗi tạm thời thì chờ base_s * 2^(lần-1) + 0..1 giây ngẫu nhiên rồi chạy lại, tối đa `attempts` lần.
    [03/10 3P] attempts 4 -> 6 (tổng chờ ~62s) + jitter: partner và 3rd-party chạy song song (ForEach FL_00) có thể ghi
    cùng bảng control; jitter tránh 2 run thử lại đúng cùng lúc. Partner: chỉ thêm lượt thử lại, logic không đổi.

    Input : fn — hàm không tham số ; label — tên thao tác để ghi log ; attempts — số lần chạy tối đa ;
            base_s — thời gian chờ lần đầu (giây)
    Output: giá trị fn() trả về. Hết lượt hoặc lỗi không tạm thời -> raise lỗi gốc
    Ví dụ : with_retry(lambda: df.write.mode("append").saveAsTable(t), "ctrl_log_table_run")
            -> xung đột lần 1: chờ ~2s, lần 2: ~4s, ... lần 5: ~32s, lần 6 lỗi thì raise
    """
    for i in range(1, attempts + 1):
        try:
            return fn()
        except Exception as e:
            if i == attempts or not is_transient(e):
                raise
            wait = base_s * 2 ** (i - 1) + random.uniform(0, 1)
            log(f"{label}: lỗi tạm thời, thử lại sau {wait:.0f}s ({i}/{attempts - 1})", "WARN")
            time.sleep(wait)


## 6. convert_utils — Chuyển kiểu dữ liệu theo `convert_rule`

Giá trị lấy từ JSON luôn là chuỗi; hàm trong nhóm này sinh biểu thức Spark SQL để chuyển chuỗi đó sang kiểu đích.
Đọc là chuỗi trước rồi mới chuyển giúp giữ được giá trị gốc để phát hiện giá trị bị mất do sai định dạng.

| Hàm / hằng số | Mục đích |
|---|---|
| `CAST_FN` | `CAST` khi ANSI tắt (Runtime 1.3, kết quả giống notebook cũ), `TRY_CAST` khi ANSI bật (Runtime 2.0, lỗi chuyển kiểu không làm dừng job) |
| `CONVERT_RULES` | Mẫu SQL cho từng rule: `NONE`, `EPOCH_S_TS`, `EPOCH_MS_TS`, `EPOCH_US_TS`, `DATE_DAYS`, `DECIMAL_BASE64`, `LOWER_TRIM` [03/10 3P] |
| `decimal_base64_sql(col, data_type)` | Giải mã decimal Debezium (base64 của số nguyên chưa chia scale) |
| `convert_sql(raw_col, data_type, convert_rule)` | Biểu thức chuyển 1 cột chuỗi sang kiểu đích |
| `DERIVED_RULES`, `derived_sql(raw_cols, rule)` | **[03/10 3P]** Khoá suy ra từ nhiều giá trị (json_path dạng `p1,p2,...`); 1 phần NULL/rỗng → NULL (thiếu khoá) |

Rule khoá suy ra (giữ đúng công thức notebook cũ để đối soát):

| Rule | Công thức | Dùng cho |
|---|---|---|
| `HASH_MD5_UUID` | `md5(concat(p1, p2, ...))` định dạng UUID 8-4-4-4-12 | `poi_id` (= `poi_group_id` cũ) |
| `HASH_SHA256` | `sha2(concat(p1, p2, ...), 256)` | `review_id` (= `source_review_id` cũ) |
| `HASH_SHA256_PIPE` | `sha2(concat_ws('\|', trim(p1), trim(p2), ...), 256)` | `media_dedup_key` (giống cũ) |


In [ ]:
# =============================================================================
# 6. convert_utils
# =============================================================================
CAST_FN = "TRY_CAST" if spark.conf.get("spark.sql.ansi.enabled", "false").lower() == "true" else "CAST"

# {col}: cột chuỗi gốc (đã quote), {dt}: kiểu đích, {cast}: CAST_FN. DECIMAL_BASE64 dựng riêng (cần scale)
CONVERT_RULES = {
    "NONE":           "{cast}({col} AS {dt})",
    "EPOCH_S_TS":     "{cast}(timestamp_seconds({cast}({col} AS BIGINT)) AS {dt})",
    "EPOCH_MS_TS":    "{cast}(timestamp_millis({cast}({col} AS BIGINT)) AS {dt})",
    "EPOCH_US_TS":    "{cast}(timestamp_micros({cast}({col} AS BIGINT)) AS {dt})",
    "DATE_DAYS":      "{cast}(date_add(DATE'1970-01-01', {cast}({col} AS INT)) AS {dt})",
    "DECIMAL_BASE64": None,
    "LOWER_TRIM":     "{cast}(lower(trim({col})) AS {dt})",       # [03/10 3P] chuẩn hoá khoá chuỗi: ' VI ' -> 'vi'
}

# [03/10 3P] Khoá suy ra từ nhiều đường dẫn. Chỉ dùng cho bảng snapshot, kiểu đích string
DERIVED_RULES = ("HASH_MD5_UUID", "HASH_SHA256", "HASH_SHA256_PIPE")


def decimal_base64_sql(col: str, data_type: str) -> str:
    """Biểu thức SQL giải mã decimal của Debezium (decimal.handling.mode = precise):
    chuỗi base64 của số nguyên chưa chia scale, big-endian, bù 2 -> decimal(p,s).
    Chỉ dùng hàm Spark có sẵn (không UDF). Hỗ trợ tới 15 byte; vượt kiểu đích -> NULL.

    Input : col — tên cột chuỗi đã quote, vd "`_raw__total_payment`" ; data_type — "decimal(p,s)"
    Output: str biểu thức SQL
    Ví dụ : 'CMHohyA=' -> byte 08 C1 E8 87 20 -> 37612980000 -> decimal(19,4) = 3761298.0000
            'C319PYA=' -> 49350000000 -> 4935000.0000 ; '/w==' (0xFF) -> -1 -> -0.0001
    """
    scale = int(re.fullmatch(r"decimal\(\d+,(\d+)\)", data_type).group(1))
    hx, nbytes = f"hex(unbase64({col}))", f"length(unbase64({col}))"
    # conv() chỉ đúng tới 64 bit -> tách 7 byte cuối và phần byte đầu, ghép lại bằng DECIMAL(38,0)
    low = f"CAST(conv(right({hx}, 14), 16, 10) AS DECIMAL(38,0))"
    high = (f"CAST(CASE WHEN length({hx}) > 14 THEN conv(left({hx}, length({hx}) - 14), 16, 10) "
            f"ELSE '0' END AS DECIMAL(38,0))")
    unsigned = f"({high} * CAST('{2 ** 56}' AS DECIMAL(38,0)) + {low})"
    # Byte đầu >= 0x80 là số âm (bù 2) -> trừ 2^(8 * số byte); hằng số viết sẵn để chính xác tuyệt đối
    two_pow = "CASE " + " ".join(f"WHEN {nbytes} = {i} THEN CAST('{2 ** (8 * i)}' AS DECIMAL(38,0))"
                                 for i in range(1, 16)) + " END"
    signed = (f"(CASE WHEN CAST(conv(left({hx}, 2), 16, 10) AS INT) >= 128 "
              f"THEN {unsigned} - {two_pow} ELSE {unsigned} END)")
    return f"CAST({signed} / CAST('{10 ** scale}' AS DECIMAL(38,0)) AS {data_type})"


def convert_sql(raw_col: str, data_type: str, convert_rule: str) -> str:
    """Biểu thức SQL chuyển 1 cột chuỗi sang kiểu đích theo convert_rule.

    Input : raw_col — tên cột chuỗi (chưa quote), vd "_raw__created_at"
            data_type — kiểu đích đã chuẩn hoá (canonical_type), vd "timestamp"
            convert_rule — 1 khoá của CONVERT_RULES
    Output: str biểu thức, dùng với F.expr(...)
    Ví dụ : convert_sql("_raw__id", "bigint", "NONE")
              -> "CAST(`_raw__id` AS bigint)"                       ("123" -> 123)
            convert_sql("_raw__last_verify_at", "timestamp", "EPOCH_S_TS")
              -> "CAST(timestamp_seconds(CAST(`_raw__last_verify_at` AS BIGINT)) AS timestamp)"
                                                                    ("1726000000" -> 2024-09-10 20:26:40 UTC)
            convert_sql("_raw__d", "date", "DATE_DAYS")              ("19980" -> 2024-09-14)
    """
    col = f"`{raw_col}`"
    if convert_rule == "DECIMAL_BASE64":
        return decimal_base64_sql(col, data_type)
    return CONVERT_RULES[convert_rule].format(cast=CAST_FN, col=col, dt=data_type)


def derived_sql(raw_cols: list, rule: str) -> str:
    """[03/10 3P] Biểu thức SQL khoá suy ra từ nhiều cột chuỗi. Có phần NULL hoặc rỗng (sau trim) -> NULL,
    để dòng bị tính là thiếu khoá thay vì sinh khoá sai (vd hash của "google|123|").

    Input : raw_cols — tên cột chuỗi (chưa quote), theo thứ tự trong json_path ; rule — 1 phần tử DERIVED_RULES
    Output: str biểu thức, dùng với F.expr(...)
    Ví dụ : derived_sql(["_raw__poi_id__0", "_raw__poi_id__1"], "HASH_MD5_UUID")
              ("google", "ChIJ123") -> "83c577de-0491-adff-5ce0-a78f7b7dc193"
            derived_sql([...], "HASH_SHA256_PIPE")   (" google", "1", "uri ") = sha2("google|1|uri", 256)
            1 phần NULL hoặc "  " -> NULL
    """
    cols = [f"`{c}`" for c in raw_cols]
    missing = " OR ".join(f"{c} IS NULL OR trim({c}) = ''" for c in cols)
    if rule == "HASH_MD5_UUID":
        value = (f"regexp_replace(md5(concat({', '.join(cols)})), "
                 "'^(.{8})(.{4})(.{4})(.{4})(.{12})$', '$1-$2-$3-$4-$5')")
    elif rule == "HASH_SHA256":
        value = f"sha2(concat({', '.join(cols)}), 256)"
    elif rule == "HASH_SHA256_PIPE":
        value = f"sha2(concat_ws('|', {', '.join(f'trim({c})' for c in cols)}), 256)"
    else:
        raise ValueError(f"rule khoá suy ra không hỗ trợ: {rule}")
    return f"CASE WHEN {missing} THEN CAST(NULL AS STRING) ELSE {value} END"


## 7. json_utils — Parse chuỗi JSON

| Hàm / hằng số | Mục đích |
|---|---|
| `JSON_OPTIONS` | Tuỳ chọn parse: chấp nhận ký tự xuống dòng/tab chưa escape và nháy đơn, giống `get_json_object` (notebook cũ) |
| `CORRUPT_COL` | Tên trường `__corrupt`: `from_json` ghi chuỗi gốc vào đây khi JSON không hợp lệ |
| `parse_json_strings(json_col, paths)` | Parse 1 lần, lấy nhiều trường cấp 1, mọi trường dạng chuỗi |
| `is_json_corrupt(parsed)` | Cờ JSON hỏng (bị cắt, không phải object…) |

Vì sao parse 1 lần bằng `from_json` thay vì `get_json_object` từng cột: `get_json_object` đọc lại cả chuỗi JSON cho mỗi cột (bảng 30 cột = 30 lần).


In [ ]:
# =============================================================================
# 7. json_utils
# =============================================================================
JSON_OPTIONS = {"allowUnquotedControlChars": "true", "allowSingleQuotes": "true"}
CORRUPT_COL = "__corrupt"


def parse_json_strings(json_col: Column, paths) -> Column:
    """Parse cột chuỗi JSON thành struct; mọi trường lấy dạng chuỗi (số, object con giữ nguyên văn bản JSON).
    Kèm trường __corrupt chứa chuỗi gốc nếu JSON không hợp lệ.

    Input : json_col — Column chuỗi JSON, vd F.col("after") ; paths — tên trường cấp 1, vd ["id", "meta"]
    Output: Column struct<id:string, meta:string, __corrupt:string>. Lấy trường: parsed.getField("id")
    Ví dụ : '{"id": 7, "meta": {"a": 1}}' -> {id: "7", meta: '{"a":1}', __corrupt: NULL}
            '{"id": "P8", "na'          -> {id: NULL, meta: NULL, __corrupt: '{"id": "P8", "na'}
            '' hoặc '   '               -> NULL   (chuỗi rỗng: KHÔNG có __corrupt, xem is_json_corrupt)
            NULL                        -> NULL
    """
    schema = StructType([StructField(p, StringType()) for p in sorted(set(paths))]
                        + [StructField(CORRUPT_COL, StringType())])
    return F.from_json(json_col, schema, {**JSON_OPTIONS, "columnNameOfCorruptRecord": CORRUPT_COL})


def is_json_corrupt(parsed: Column, json_col: Optional[Column] = None) -> Column:
    """Cờ JSON không parse được, dùng với kết quả của parse_json_strings.

    Input : parsed   — Column struct từ parse_json_strings
            json_col — (tuỳ chọn) Column chuỗi gốc đã đưa vào parse. Truyền vào để bắt cả chuỗi rỗng/khoảng trắng:
                       from_json trả struct NULL mà không ghi __corrupt, nên chỉ nhìn parsed sẽ không thấy.
    Output: Column boolean (True = JSON hỏng; đầu vào NULL -> False)
    Ví dụ : '{"id":'  -> True ; '' -> False nếu không truyền json_col, True nếu có ; '{}' -> False ; NULL -> False
            df.where(is_json_corrupt(F.col("_p"), F.col("after")))  -> các dòng payload hỏng
    """
    corrupt = parsed.getField(CORRUPT_COL).isNotNull()
    if json_col is None:
        return corrupt
    return corrupt | (json_col.isNotNull() & parsed.isNull())


## 8. data_models — Cấu trúc dữ liệu

| Class | Chứa gì | Ghi chú |
|---|---|---|
| `ConfigError` | Lỗi cấu hình | Raise khi cấu hình sai, trước khi ghi bất cứ thứ gì |
| `CastNullError` | Có giá trị thành NULL khi chuyển kiểu (`cast_null_policy = FAIL`) | Bảng FAILED trước MERGE, không retry; watermark giữ nguyên |
| `ColumnSpec` | 1 cột đích: tên, `json_path`, kiểu, `convert_rule`, khoá, TOAST, thứ tự | Bất biến (`frozen`) vì nhiều luồng cùng đọc |
| `TableSpec` | 1 bảng đích: schema/tên, bảng nguồn trong raw, `priority`, danh sách cột | Bất biến |
| `RunContext` | Thông tin 1 lần chạy: id, tham số, chế độ chạy, watermark, version raw | Luồng chính điền dần; luồng con chỉ đọc |
| `Profile` | Số đếm sau khi đọc raw và `watermark_to` | |
| `TableResult` | Kết quả 1 bảng: trạng thái, số đếm, lỗi, DataFrame state/reject chờ commit, cache cần giải phóng | Khớp các cột của `ctrl_log_table_run` |
| `RunOutcome` | **[03/10 3P]** Kết quả cả lần chạy do phần xử lý riêng của notebook điền dần (trạng thái, profile, kết quả bảng, cache) | Dùng với `run_controlled` (§14) |


In [ ]:
# =============================================================================
# 8. data_models
# =============================================================================
class ConfigError(Exception):
    """Cấu hình sai: dừng ngay, chưa ghi gì."""


class CastNullError(Exception):
    """Giá trị có ở raw nhưng thành NULL khi chuyển kiểu (sai data_type/convert_rule hoặc dữ liệu nguồn lỗi).
    cast_null_policy = FAIL: bảng FAILED trước MERGE (không retry vì không phải lỗi tạm thời)."""

CAST_NULL_POLICIES = ("FAIL", "WARN")


@dataclass(frozen=True)
class ColumnSpec:
    """1 cột của bảng đích (1 dòng active trong ctrl_cfg_schema_registry).

    Ví dụ : ColumnSpec(name="id", json_path="id", data_type="string", convert_rule="NONE",
                       is_key=True, key_order=1, is_toast=False, column_order=1)
    """
    name: str
    json_path: str
    data_type: str              # kiểu Spark đã chuẩn hoá, vd bigint, decimal(19,4)
    convert_rule: str
    is_key: bool
    key_order: Optional[int]
    is_toast: bool
    column_order: int

    @property
    def raw_col(self) -> str:
        """Tên cột nội bộ chứa chuỗi gốc lấy từ JSON. Vd "id" -> "_raw__id"."""
        return f"_raw__{self.name}"

    @property
    def is_derived(self) -> bool:
        """[03/10 3P] Cột khoá suy ra từ nhiều giá trị (convert_rule thuộc DERIVED_RULES)."""
        return self.convert_rule in DERIVED_RULES

    @property
    def paths(self) -> list:
        """[03/10 3P] Các đường dẫn trong json_path. Vd "_doc.source_name,_doc.source_id" -> 2 phần; "id" -> ["id"]."""
        return [p.strip() for p in self.json_path.split(",")]

    @property
    def part_cols(self) -> list:
        """[03/10 3P] Tên cột nội bộ chứa chuỗi gốc của từng phần (cột suy ra). Vd poi_id -> ["_raw__poi_id__0", ...]."""
        return [f"{self.raw_col}__{i}" for i in range(len(self.paths))]

    @property
    def can_cast_null(self) -> bool:
        """Cột có thể mất giá trị khi chuyển kiểu không. Cột chuỗi giữ nguyên (NONE, LOWER_TRIM) và cột khoá suy ra
        (NULL nghĩa là thiếu khoá, tính ở bước reject) thì không.
        [03/10 3P] thêm LOWER_TRIM, cột suy ra; cột partner (NONE + kiểu khác) không đổi."""
        if self.is_derived:
            return False
        return not (self.data_type == "string" and self.convert_rule in ("NONE", "LOWER_TRIM"))


@dataclass(frozen=True)
class TableSpec:
    """1 bảng đích và nguồn của nó trong raw.

    Ví dụ : TableSpec(trg_schema="lh_vv_silver.dbo", trg_tbl="slv_products", src_object_schema="public",
                      src_object="products", priority=1, columns=(ColumnSpec(...), ...))
    """
    trg_schema: str
    trg_tbl: str
    src_object_schema: Optional[str]
    src_object: str
    priority: int
    columns: tuple
    # [03/10 3P] 3 trường mới, có mặc định = hành vi partner (CDC)
    load_mode: str = "CDC"              # LOAD_MODES
    align_path: Optional[str] = None    # DOC_ARRAY / LANG_ARRAY: mảng (tính từ gốc payload) ghép theo vị trí -> _align.*
    dedup_order: tuple = ()             # ((tên cột, desc), ...): chọn 1 dòng khi 1 tài liệu có nhiều dòng cùng khoá

    @property
    def fq_name(self) -> str:
        """Tên đầy đủ bảng đích. Vd "lh_vv_silver.dbo.slv_products"."""
        return f"{self.trg_schema}.{self.trg_tbl}"

    @property
    def is_snapshot(self) -> bool:
        """[03/10 3P] Bảng của nguồn snapshot (3rd-party), không phải CDC."""
        return self.load_mode in SNAPSHOT_MODES

    @property
    def tech_columns(self) -> tuple:
        """[03/10 3P] Cột kỹ thuật của bảng đích theo load_mode: CDC -> TECH_COLUMNS ; snapshot -> SNAPSHOT_TECH_COLUMNS."""
        return SNAPSHOT_TECH_COLUMNS if self.is_snapshot else TECH_COLUMNS

    @property
    def key_cols(self) -> list:
        """Cột khoá theo key_order. Vd slv_business_service_i18ns -> ["business_service_id", "locale"]."""
        return [c.name for c in sorted((c for c in self.columns if c.is_key), key=lambda c: c.key_order)]

    @property
    def toast_cols(self) -> list:
        """Cột có thể nhận giá trị TOAST thay thế (không tính cột khoá)."""
        return [c.name for c in self.columns if c.is_toast and not c.is_key]

    @property
    def src_ref(self) -> tuple:
        """(schema, bảng) nguồn. Vd ("public", "products")."""
        return (self.src_object_schema, self.src_object)

    def matches(self, obj_schema: Optional[str], obj: Optional[str]) -> bool:
        """Bảng nguồn (obj_schema, obj) có thuộc bảng đích này không. src_object_schema NULL = khớp mọi schema.
        Vd spec products/public: matches("public", "products") -> True ; matches("other", "products") -> False
        """
        return obj == self.src_object and (self.src_object_schema is None or obj_schema == self.src_object_schema)


@dataclass
class RunContext:
    """Thông tin 1 lần chạy notebook. Các trường sau dấu '=' được điền dần trong lúc chạy."""
    exec_id: str                        # uuid mỗi lần notebook chạy — khoá của ctrl_log_run
    run_id: str                         # RunId của pipeline (chạy tay = exec_id)
    pl_name: str
    nb_name: str
    src_schema: str                     # vd lh_vv_bronze.dbo
    src_tbl: str                        # vd partner_raw_data
    run_mode: str                       # INCREMENTAL | FULL_RELOAD | DRY_RUN (nhãn ghi log)
    table_filter: list
    overlap_minutes: int                # không còn dùng (02/10: bỏ kiểu đọc TIME); giữ để tương thích tham số/run log
    max_parallel: int
    max_retries: int
    full_reload: bool
    dry_run: bool
    stop_on_failure: bool
    running_timeout_minutes: int
    run_params: str                     # JSON tham số, ghi vào run log
    started_at: datetime
    watermark_id: Optional[str] = None
    watermark_column: Optional[str] = None
    watermark_from: Optional[datetime] = None
    src_version_from: Optional[int] = None     # last_src_version trước khi chạy
    src_version_to: Optional[int] = None       # version raw đã chốt
    src_table_id_from: Optional[str] = None    # last_src_table_id trước khi chạy (Delta table id của raw)
    src_table_id: Optional[str] = None         # Delta table id hiện tại của raw (pin_snapshot)
    src_detail: Optional[dict] = None          # DESCRIBE DETAIL raw lúc chốt (location, minReaderVersion...)
    allow_full_scan: bool = False              # cho phép đọc FULL khi không đọc theo version được
    read_mode: Optional[str] = None            # VERSION | FULL — cách đã đọc raw (read_incremental)
    read_note: Optional[str] = None            # chi tiết: khoảng version, số file, lý do đọc FULL
    cast_null_policy: str = "FAIL"             # FAIL: bảng có giá trị thành NULL khi chuyển kiểu -> FAILED ; WARN: chỉ cảnh báo
    lock_mode: Optional[str] = None            # WATERMARK (khoá trên dòng watermark) | RUN_LOG (bảng chưa có cột khoá)

    @property
    def raw_fq(self) -> str:
        """Tên đầy đủ bảng raw. Vd "lh_vv_bronze.dbo.partner_raw_data"."""
        return f"{self.src_schema}.{self.src_tbl}"

    @property
    def updates_watermark(self) -> bool:
        """Chỉ lần chạy incremental đầy đủ (không dry_run, full_reload, table_filter) mới được đụng tới watermark."""
        return not (self.dry_run or self.full_reload or self.table_filter)


@dataclass
class Profile:
    """Số đếm sau khi đọc + phân loại raw."""
    read_rows: int = 0
    valid_rows: int = 0
    ignored_rows: int = 0
    rejected_rows: int = 0
    ignored_detail: dict = field(default_factory=dict)   # {reason: {"schema.bảng": n}}
    valid_by_src: dict = field(default_factory=dict)     # {(schema, bảng): n}
    watermark_to: Optional[datetime] = None              # max thời điểm ingest của phần đã đọc

    def events_for(self, spec: TableSpec) -> int:
        """Số event hợp lệ thuộc bảng đích `spec`. Vd 0 -> bảng NO_DATA, không cần chạy."""
        return sum(n for (s, o), n in self.valid_by_src.items() if spec.matches(s, o))


@dataclass
class TableResult:
    """Kết quả xử lý 1 bảng đích trong 1 lần chạy."""
    spec: TableSpec
    wave: int = 0
    status: str = "PENDING"             # SUCCESS | FAILED | NO_DATA | SKIPPED
    attempt_no: int = 0
    input_rows: int = 0
    stale_rows: int = 0
    applicable_rows: int = 0
    entity_rows: int = 0
    upsert_event_rows: int = 0
    delete_event_rows: int = 0
    inserted_rows: int = 0
    updated_rows: int = 0
    cast_null_rows: int = 0
    rejected_rows: int = 0
    cast_null_detail: dict = field(default_factory=dict)   # {cột: số giá trị thành NULL khi chuyển kiểu}
    started_at: Optional[datetime] = None
    ended_at: Optional[datetime] = None
    error_message: Optional[str] = None
    state_df: Optional[DataFrame] = None                   # chờ commit vào ctrl_cdc_state
    reject_df: Optional[DataFrame] = None                  # chờ commit vào ctrl_cdc_reject
    cached: list = field(default_factory=list)             # DataFrame đã persist, giải phóng sau commit

    @property
    def duration_ms(self) -> Optional[int]:
        """Thời gian xử lý (ms); None nếu chưa có đủ started_at/ended_at."""
        if self.started_at and self.ended_at:
            return int((self.ended_at - self.started_at).total_seconds() * 1000)
        return None

    def release(self) -> None:
        """Giải phóng mọi DataFrame đã persist của bảng này."""
        for df in self.cached:
            df.unpersist()
        self.cached = []


@dataclass
class RunOutcome:
    """[03/10 3P] Kết quả cả lần chạy. Phần xử lý riêng của notebook (body của run_controlled) điền dần;
    lỗi giữa chừng thì run_controlled vẫn dùng phần đã điền để ghi run log / watermark."""
    status: str = "FAILED"
    error: Optional[str] = None
    profile: Optional[Profile] = None
    results: list = field(default_factory=list)            # list[TableResult]
    cached: list = field(default_factory=list)             # DataFrame đã persist, giải phóng cuối run
    extra: dict = field(default_factory=dict)              # thêm vào JSON tóm tắt (exit value)


## 9. config_utils — Nạp và kiểm tra cấu hình

| Hàm | Mục đích |
|---|---|
| `build_table_specs(pc_rows, reg_rows, table_filter, allowed_modes)` | Hàm thuần (không đọc bảng): ghép `pipeline_config` + `registry` thành danh sách `TableSpec`, kiểm tra toàn bộ, gom mọi lỗi rồi raise 1 lần |
| `load_table_specs(ctx, allowed_modes)` | Đọc 2 bảng cấu hình của pipeline + nguồn, gọi `build_table_specs` |
| `table_has(table, *cols)` | **[03/10 3P]** Bảng control đã có các cột thêm sau chưa (dùng chung cho watermark, pipeline_config) |
| `snapshot_path_error(path, mode, src_object, align_path)` | **[03/10 3P]** Lý do 1 `json_path` của bảng snapshot không hợp lệ, hoặc None |
| `parse_dedup_order(text)` | **[03/10 3P]** `"is_original DESC, created_at DESC"` → `(("is_original", True), ("created_at", True))` |

Cấu hình sai thì dừng ngay, trước khi ghi bất cứ thứ gì; gom đủ lỗi để sửa 1 lượt.

**[03/10 3P]** Kiểm tra thêm theo `load_mode`:

| Bảng | Kiểm tra |
|---|---|
| CDC (partner) | Như cũ; thêm: `align_path` / `dedup_order` phải NULL, không dùng rule `HASH_*` |
| Snapshot | `src_object` là `$` hoặc đường dẫn (`$` không dùng với `*_ARRAY`); `json_path` theo bảng ở §1; `HASH_*` và `LOWER_TRIM` chỉ cho kiểu string; `is_toast` = false; `align_path` chỉ cho `*_ARRAY`; `dedup_order` chỉ gồm cột của bảng (hoặc `_pos`) |


In [ ]:
# =============================================================================
# 9. config_utils
# =============================================================================
def table_has(table: str, *cols: str) -> bool:
    """[03/10 3P] Bảng đã có đủ các cột này chưa (cột thêm dần bằng ALTER TABLE).

    Input : table — tên đầy đủ ; cols — tên cột
    Output: bool
    Ví dụ : table_has(T_PIPELINE_CONFIG, "load_mode") -> False khi chưa chạy NB_CREATE_DDL bản 03/10
    """
    existing = {c.lower() for c in spark.table(table).columns}
    return all(c.lower() in existing for c in cols)


def snapshot_path_error(path: str, mode: str, src_object: Optional[str], align_path: Optional[str]) -> Optional[str]:
    """[03/10 3P] Kiểm tra 1 đường dẫn (1 phần của json_path) của bảng snapshot theo bảng "json_path" ở §1.

    Input : path — vd "address_line1", "_doc.source_name", "_align.time", "$" ; mode — load_mode ;
            src_object — khối nguồn của bảng ; align_path — mảng ghép theo vị trí (hoặc None)
    Output: None nếu hợp lệ, ngược lại lý do (str)
    Ví dụ : snapshot_path_error("_lang", "DOC", "poi_address", None) -> "_lang chỉ dùng với LANG / LANG_ARRAY"
            snapshot_path_error("booking_links.agoda", "DOC", "poi_contact", None) -> None
    """
    if path == "$":
        return "'$' (cả khối) không dùng khi src_object = '$' (gốc payload)" if mode == "DOC" and src_object == "$" else None
    if path == "_lang":
        return None if mode in ("LANG", "LANG_ARRAY") else "_lang chỉ dùng với LANG / LANG_ARRAY"
    if path == "_pos":
        return None if mode in ("DOC_ARRAY", "LANG_ARRAY") else "_pos chỉ dùng với DOC_ARRAY / LANG_ARRAY"
    if path == "_langs":
        return None
    head, _, rest = path.partition(".")
    if head == "_doc":
        return None if is_identifier(rest) else "_doc.<cột raw> không hợp lệ"
    if head in ("_root", "_align"):
        if not JSON_PATH_RE.fullmatch(rest or ""):
            return f"{head}.<đường dẫn> không hợp lệ"
        if head == "_align" and not align_path:
            return "_align.* cần align_path trong ctrl_mng_pipeline_config"
        return None
    if head in PATH_SCOPES:
        return f"tiền tố {head} không dùng dạng '{path}'"
    return None if JSON_PATH_RE.fullmatch(path) else "đường dẫn không hợp lệ"


def parse_dedup_order(text: Optional[str]) -> tuple:
    """[03/10 3P] Phân tích ctrl_mng_pipeline_config.dedup_order: "<cột> [ASC|DESC], ...". Thiếu chiều = ASC.

    Input : text — str | None
    Output: tuple((tên cột, desc: bool), ...) ; rỗng/None -> () ; sai cú pháp -> ValueError
    Ví dụ : parse_dedup_order("is_original DESC, created_at") -> (("is_original", True), ("created_at", False))
    """
    items = []
    for part in split_csv(text):
        tokens = part.split()
        direction = tokens[1].upper() if len(tokens) == 2 else "ASC"
        if len(tokens) not in (1, 2) or direction not in ("ASC", "DESC") \
                or not (is_identifier(tokens[0]) or tokens[0] == "_pos"):
            raise ValueError(f"dedup_order sai cú pháp ở '{part}' (đúng: <cột> [ASC|DESC], ...)")
        items.append((tokens[0], direction == "DESC"))
    return tuple(items)


def build_table_specs(pc_rows: list, reg_rows: list, table_filter: list, allowed_modes: Optional[tuple] = None) -> tuple:
    """Ghép và kiểm tra cấu hình thành danh sách TableSpec.

    Kiểm tra: trùng bảng đích; table_filter có tên lạ; bảng không có cột active; 1 bảng đích chỉ 1 bảng nguồn;
    tên cột/json_path/convert_rule/data_type hợp lệ; DECIMAL_BASE64 chỉ cho decimal; không trùng cột kỹ thuật;
    không trùng tên/column_order; có khoá và key_order không trùng.
    [03/10 3P] Thêm: load_mode hợp lệ và thuộc allowed_modes; kiểm tra riêng cho bảng snapshot (bảng ở markdown §9).

    Input : pc_rows — list[dict] từ ctrl_mng_pipeline_config (trg_schema, trg_tbl, priority; [03/10 3P] load_mode,
                      align_path, dedup_order — thiếu key = NULL), đã lọc pl_name/nguồn/active
            reg_rows — list[dict] mọi dòng ctrl_cfg_schema_registry của nguồn (kể cả is_active = false)
            table_filter — list[str] tên trg_tbl cần chạy, [] = tất cả (không phân biệt hoa thường)
            allowed_modes — [03/10 3P] tuple load_mode notebook xử lý được; None = không giới hạn
    Output: (specs, configured)
            specs — list[TableSpec] sắp theo (priority, trg_tbl)
            configured — set[(src_object_schema, src_object)] mọi bảng nguồn có trong registry
    Ví dụ : pc_rows=[{"trg_schema": "lh_vv_silver.dbo", "trg_tbl": "slv_products", "priority": 1}]
            reg_rows=[{"src_object_schema": "public", "src_object": "products", "trg_schema": "lh_vv_silver.dbo",
                       "trg_tbl": "slv_products", "trg_column": "id", "json_path": "id", "data_type": "string",
                       "convert_rule": "NONE", "is_key": True, "key_order": 1, "is_toast": False,
                       "column_order": 1, "is_active": True}]
            -> ([TableSpec(slv_products, 1 cột)], {("public", "products")})
            Thiếu cột khoá -> ConfigError("Cấu hình sai:\n  - lh_vv_silver.dbo.slv_products: không có cột khoá (is_key)")
    """
    errors = []

    # 1. pipeline_config: bỏ dòng trùng (báo lỗi), áp table_filter
    unique_rows = {}
    for r in pc_rows:
        key = (r["trg_schema"], r["trg_tbl"])
        if key in unique_rows:
            errors.append(f"pipeline_config trùng bảng đích {key[0]}.{key[1]}")
        unique_rows.setdefault(key, r)
    pc_rows = list(unique_rows.values())
    if not pc_rows:
        errors.append("pipeline_config không có bảng active cho pipeline/nguồn này")

    wanted = {t.strip().lower() for t in table_filter if t.strip()}
    if wanted:
        unknown = wanted - {r["trg_tbl"].lower() for r in pc_rows}
        if unknown:
            errors.append(f"table_filter có bảng không active trong pipeline_config: {sorted(unknown)}")
        pc_rows = [r for r in pc_rows if r["trg_tbl"].lower() in wanted]

    # 2. registry: nhóm cột active theo bảng đích
    configured = {(r["src_object_schema"], r["src_object"]) for r in reg_rows if r["src_object"]}
    reg_by_tbl: dict = {}
    for r in reg_rows:
        if r["is_active"]:
            reg_by_tbl.setdefault((r["trg_schema"], r["trg_tbl"]), []).append(r)

    # 3. từng bảng đích
    specs = []
    for p in sorted(pc_rows, key=lambda r: (r["priority"] if r["priority"] is not None else 0, r["trg_tbl"])):
        tbl = f"{p['trg_schema']}.{p['trg_tbl']}"
        tbl_errors = []
        if not is_fq_name(p["trg_schema"]) or not is_identifier(p["trg_tbl"]):
            tbl_errors.append("tên schema/bảng đích không hợp lệ")
        if p["priority"] is None:
            tbl_errors.append("priority NULL")

        # [03/10 3P] load_mode + 2 tuỳ chọn của bảng snapshot (NULL = CDC, giữ hành vi partner)
        mode = str(p.get("load_mode") or "CDC").strip().upper()
        align = str(p.get("align_path") or "").strip() or None
        dedup_text = str(p.get("dedup_order") or "").strip() or None
        if mode not in LOAD_MODES:
            tbl_errors.append(f"load_mode không hỗ trợ '{p.get('load_mode')}' (nhận {', '.join(LOAD_MODES)})")
        elif allowed_modes is not None and mode not in allowed_modes:
            tbl_errors.append(f"load_mode {mode}: notebook này chỉ xử lý {', '.join(allowed_modes)}")
        snapshot = mode in SNAPSHOT_MODES
        if not snapshot and (align or dedup_text):
            tbl_errors.append("align_path / dedup_order chỉ dùng cho bảng snapshot (load_mode khác CDC)")
        if align and (mode not in ("DOC_ARRAY", "LANG_ARRAY") or not JSON_PATH_RE.fullmatch(align)):
            tbl_errors.append(f"align_path '{align}' không hợp lệ (chỉ cho DOC_ARRAY / LANG_ARRAY, dạng a.b)")

        rows = reg_by_tbl.get((p["trg_schema"], p["trg_tbl"]), [])
        if not rows:
            errors.append(f"{tbl}: chưa có cột active trong ctrl_cfg_schema_registry")
            continue
        sources = {(r["src_object_schema"], r["src_object"]) for r in rows}
        if len(sources) != 1:
            tbl_errors.append(f"nhiều bảng nguồn cho 1 bảng đích: {sorted(map(str, sources))}")
        src_obj = next(iter(sources))[1] if len(sources) == 1 else None
        if snapshot and src_obj is not None:
            if src_obj == "$" and mode in ("DOC_ARRAY", "LANG_ARRAY"):
                tbl_errors.append(f"src_object '$' không dùng với {mode} (cần đường dẫn tới mảng)")
            elif src_obj != "$" and not JSON_PATH_RE.fullmatch(src_obj):
                tbl_errors.append(f"src_object '{src_obj}' không hợp lệ (dạng $ hoặc a.b)")

        tech_names = {n for n, _ in (SNAPSHOT_TECH_COLUMNS if snapshot else TECH_COLUMNS)}
        columns = []
        for r in sorted(rows, key=lambda r: (r["column_order"] is None, r["column_order"])):
            col = r["trg_column"] or ""
            if not is_identifier(col):
                tbl_errors.append(f"tên cột không hợp lệ '{col}'")
                continue
            if col.lower() in tech_names:
                tbl_errors.append(f"cột '{col}' trùng cột kỹ thuật")
            rule = (r["convert_rule"] or "NONE").upper()
            if rule not in CONVERT_RULES and rule not in DERIVED_RULES:
                tbl_errors.append(f"{col}: convert_rule không hỗ trợ '{r['convert_rule']}'")
            json_path = r["json_path"] or ""
            if not snapshot:
                if rule in DERIVED_RULES:
                    tbl_errors.append(f"{col}: convert_rule {rule} chỉ dùng cho bảng snapshot")
                if not JSON_PATH_RE.fullmatch(json_path):
                    tbl_errors.append(f"{col}: json_path không hợp lệ '{r['json_path']}'")
            else:                                                            # [03/10 3P]
                parts = [x.strip() for x in json_path.split(",")]
                if len(parts) > 1 and rule not in DERIVED_RULES:
                    tbl_errors.append(f"{col}: json_path nhiều phần chỉ dùng với rule {'/'.join(DERIVED_RULES)}")
                for part in parts:
                    why = snapshot_path_error(part, mode, src_obj, align)
                    if why:
                        tbl_errors.append(f"{col}: json_path không hợp lệ '{part}' ({why})")
                if r["is_toast"]:
                    tbl_errors.append(f"{col}: is_toast chỉ dùng cho CDC")
            try:
                data_type = canonical_type(r["data_type"])
            except ValueError as e:
                tbl_errors.append(f"{col}: {e}")
                continue
            if rule == "DECIMAL_BASE64" and not data_type.startswith("decimal("):
                tbl_errors.append(f"{col}: DECIMAL_BASE64 chỉ dùng cho kiểu decimal(p,s), đang là {data_type}")
            if (rule in DERIVED_RULES or rule == "LOWER_TRIM") and data_type != "string":
                tbl_errors.append(f"{col}: {rule} chỉ dùng cho kiểu string, đang là {data_type}")
            columns.append(ColumnSpec(
                name=col, json_path=json_path, data_type=data_type, convert_rule=rule,
                is_key=bool(r["is_key"]), key_order=r["key_order"], is_toast=bool(r["is_toast"]),
                column_order=r["column_order"],
            ))

        names = [c.name.lower() for c in columns]
        dup_names = sorted({n for n in names if names.count(n) > 1})
        if dup_names:
            tbl_errors.append(f"trùng tên cột {dup_names}")
        orders = [c.column_order for c in columns]
        if None in orders or len(set(orders)) != len(orders):
            tbl_errors.append("column_order NULL hoặc trùng")
        keys = [c for c in columns if c.is_key]
        key_orders = [c.key_order for c in keys]
        if not keys:
            tbl_errors.append("không có cột khoá (is_key)")
        elif None in key_orders or len(set(key_orders)) != len(key_orders):
            tbl_errors.append("key_order NULL hoặc trùng")

        dedup = ()                                                           # [03/10 3P]
        if dedup_text:
            try:
                dedup = parse_dedup_order(dedup_text)
            except ValueError as e:
                tbl_errors.append(str(e))
            allowed_names = set(names) | ({"_pos"} if mode in ("DOC_ARRAY", "LANG_ARRAY") else set())
            bad = [n for n, _ in dedup if n.lower() not in allowed_names]
            if bad:
                tbl_errors.append(f"dedup_order có cột không thuộc bảng: {bad}")

        if tbl_errors:
            errors.extend(f"{tbl}: {e}" for e in tbl_errors)
            continue
        src_obj_schema, src_obj = next(iter(sources))
        specs.append(TableSpec(
            trg_schema=p["trg_schema"], trg_tbl=p["trg_tbl"],
            src_object_schema=src_obj_schema, src_object=src_obj,
            priority=int(p["priority"]), columns=tuple(columns),
            load_mode=mode, align_path=align, dedup_order=dedup,
        ))

    if errors:
        raise ConfigError("Cấu hình sai:\n  - " + "\n  - ".join(errors))
    return specs, configured


PC_OPTIONAL_COLUMNS = ("load_mode", "align_path", "dedup_order")      # [03/10 3P] cột mới của pipeline_config


def load_table_specs(ctx: RunContext, allowed_modes: Optional[tuple] = ("CDC",)) -> tuple:
    """Đọc cấu hình của pipeline + nguồn từ bảng control rồi kiểm tra (build_table_specs).
    Giá trị truyền bằng tham số (:pl, :ss, :st), không ghép chuỗi.
    [03/10 3P] Đọc thêm load_mode, align_path, dedup_order (bảng chưa có cột -> NULL = CDC);
    allowed_modes: load_mode notebook gọi xử lý được, mặc định ("CDC",) = partner.

    Input : ctx — RunContext (dùng pl_name, src_schema, src_tbl, table_filter) ; allowed_modes — tuple | None
    Output: (specs, configured) như build_table_specs ; cấu hình sai -> ConfigError
    Ví dụ : load_table_specs(ctx) với partner -> (23 TableSpec, 23 bảng nguồn)
            load_table_specs(ctx, SNAPSHOT_MODES) với brz_3rd_crawler_poi_stream -> (14 TableSpec, ...)
    """
    src = {"ss": ctx.src_schema, "st": ctx.src_tbl}
    optional = ", ".join(c if table_has(T_PIPELINE_CONFIG, c) else f"CAST(NULL AS STRING) AS {c}"
                         for c in PC_OPTIONAL_COLUMNS)
    pc_rows = spark.sql(f"""
        SELECT trg_schema, trg_tbl, priority, {optional}
        FROM {T_PIPELINE_CONFIG}
        WHERE pl_name = :pl AND src_schema = :ss AND src_tbl = :st AND is_active = 1
    """, args={"pl": ctx.pl_name, **src}).collect()
    reg_rows = spark.sql(f"""
        SELECT src_object_schema, src_object, trg_schema, trg_tbl, trg_column, json_path, data_type,
               convert_rule, is_key, key_order, is_toast, column_order, is_active
        FROM {T_REGISTRY}
        WHERE src_schema = :ss AND src_tbl = :st
    """, args=src).collect()
    return build_table_specs([r.asDict() for r in pc_rows], [r.asDict() for r in reg_rows], ctx.table_filter,
                             allowed_modes)


## 10. delta_utils — Thao tác bảng Delta

| Hàm | Mục đích |
|---|---|
| `current_version(table)` | Version mới nhất của bảng Delta |
| `table_detail(table)` | `DESCRIBE DETAIL`: `location`, `partitionColumns`, `minReaderVersion`… |
| `delta_added_files(detail, v_from, v_to)` | Danh sách file dữ liệu được thêm trong các commit `(v_from, v_to]`, đọc thẳng `_delta_log/<version>.json`; gặp trường hợp không an toàn → `DeltaReadUnsupported` |
| `read_delta_files(table, location, paths)` | Đọc trực tiếp các file parquet đó theo schema hiện tại của bảng |
| `target_ddl(spec)` | Câu `CREATE TABLE` bảng đích từ cấu hình + cột kỹ thuật (**[03/10 3P]** theo `spec.tech_columns`) |
| `ensure_target(spec, dry_run)` | Tạo bảng nếu chưa có; thêm cột mới cấu hình; báo lỗi nếu lệch kiểu (**[03/10 3P]** cột kỹ thuật theo `spec.tech_columns`) |
| `run_merge(source, target_fq, build_sql, view_name)` | Chạy MERGE bằng Spark SQL qua temp view, trả số dòng insert/update |
| `append_rows(table, rows)` | Append list[dict] vào bảng theo đúng schema bảng (thứ tự, kiểu cột) |


In [ ]:
# =============================================================================
# 10. delta_utils
# =============================================================================
def current_version(table: str) -> int:
    """Version mới nhất của bảng Delta.

    Input : table — tên đầy đủ, vd "lh_vv_bronze.dbo.partner_raw_data"
    Output: int
    Ví dụ : current_version("lh_vv_bronze.dbo.partner_raw_data") -> 18543
    """
    return int(DeltaTable.forName(spark, table).history(1).select("version").first()[0])


def target_ddl(spec: TableSpec, tech_columns: Optional[tuple] = None) -> str:
    """Câu CREATE TABLE cho bảng đích: cột theo column_order + cột kỹ thuật,
    bật optimizeWrite + autoCompact. Không CLUSTER BY (Runtime 1.3 không nên dùng cùng auto compaction).

    Input : spec — TableSpec ; tech_columns — ((tên, kiểu), ...) cột kỹ thuật;
            [03/10 3P] None (mặc định) = spec.tech_columns (CDC: TECH_COLUMNS như cũ ; snapshot: SNAPSHOT_TECH_COLUMNS)
    Output: str câu SQL
    Ví dụ : target_ddl(slv_products) ->
            CREATE TABLE IF NOT EXISTS `lh_vv_silver`.`dbo`.`slv_products` (
                `id` STRING, ..., `deleted` BOOLEAN, `_ingested_at` TIMESTAMP, `_source_db` STRING
            ) USING DELTA TBLPROPERTIES (...)
    """
    tech_columns = spec.tech_columns if tech_columns is None else tech_columns
    cols = [f"`{c.name}` {c.data_type.upper()}" for c in spec.columns]
    cols += [f"`{n}` {t.upper()}" for n, t in tech_columns]
    body = ",\n    ".join(cols)
    return (
        f"CREATE TABLE IF NOT EXISTS {quote_name(spec.fq_name)} (\n    {body}\n) USING DELTA\n"
        "TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true', 'delta.autoOptimize.autoCompact' = 'true')"
    )


def ensure_target(spec: TableSpec, dry_run: bool, tech_columns: Optional[tuple] = None) -> Optional[str]:
    """Chuẩn bị bảng đích trước khi MERGE.
    - Chưa có bảng -> CREATE TABLE (target_ddl)
    - Cấu hình có cột mới -> ALTER TABLE ADD COLUMNS (thêm cột chỉ cần INSERT vào registry)
    - Lệch kiểu giữa bảng và cấu hình -> ConfigError
    dry_run = True: chỉ kiểm tra, không tạo/sửa.

    Input : spec — TableSpec ; dry_run — bool ; tech_columns — cột kỹ thuật, [03/10 3P] None = spec.tech_columns
    Output: str mô tả thay đổi ("tạo bảng mới", "thêm cột ['a']") hoặc None nếu không đổi gì
    Ví dụ : bảng chưa có -> "tạo bảng mới" ; registry thêm cột "tier" -> "thêm cột ['tier']"
            bảng có id BIGINT, cấu hình id STRING -> ConfigError("lệch kiểu cột: id (bảng: bigint, cấu hình: string)")
    """
    tech_columns = spec.tech_columns if tech_columns is None else tech_columns
    if not spark.catalog.tableExists(spec.fq_name):
        if not dry_run:
            spark.sql(target_ddl(spec, tech_columns))
        return "tạo bảng mới"
    existing = {f.name.lower(): f.dataType.simpleString() for f in spark.table(spec.fq_name).schema.fields}
    wanted = [(c.name, c.data_type) for c in spec.columns] + list(tech_columns)
    mismatched = [f"{n} (bảng: {existing[n.lower()]}, cấu hình: {t})"
                  for n, t in wanted if n.lower() in existing and existing[n.lower()] != t]
    if mismatched:
        raise ConfigError(f"lệch kiểu cột: {'; '.join(mismatched)}")
    missing = [(n, t) for n, t in wanted if n.lower() not in existing]
    if missing and not dry_run:
        spark.sql(f"ALTER TABLE {quote_name(spec.fq_name)} ADD COLUMNS ("
                  + ", ".join(f"`{n}` {t.upper()}" for n, t in missing) + ")")
    return f"thêm cột {[n for n, _ in missing]}" if missing else None


def run_merge(source: DataFrame, target_fq: str, build_sql: Callable, view_name: str) -> tuple:
    """Chạy MERGE (Spark SQL) từ DataFrame nguồn vào bảng Delta, trả số dòng insert/update.
    Nguồn được đăng ký thành temp view tên `view_name` (phải duy nhất giữa các luồng), xoá view khi xong.
    Số dòng lấy từ kết quả MERGE; nếu không có thì đọc commit MERGE trong history
    (auto compaction có thể tạo commit OPTIMIZE ngay sau MERGE nên không dùng history(1)).
    Yêu cầu: mỗi bảng đích chỉ 1 luồng ghi tại một thời điểm.

    Input : source — DataFrame nguồn, đã 1 dòng/khoá
            target_fq — tên đầy đủ bảng đích, vd "lh_vv_silver.dbo.slv_products"
            build_sql(target_quoted, view_name) -> str câu MERGE
            view_name — vd "_v_826b4120_slv_products"
    Output: (inserted_rows, updated_rows)
    Ví dụ : run_merge(df, "lh_vv_silver.dbo.slv_x",
                      lambda t, v: f"MERGE INTO {t} AS t USING {v} AS s ON t.id = s.id WHEN MATCHED ...",
                      "_v_826b4120_slv_x") -> (12, 108)
    """
    version_before = current_version(target_fq)
    source.createOrReplaceTempView(view_name)
    try:
        result = spark.sql(build_sql(quote_name(target_fq), view_name)).first()
    finally:
        spark.catalog.dropTempView(view_name)

    metrics = result.asDict() if result is not None else {}
    if "num_inserted_rows" in metrics:
        return int(metrics["num_inserted_rows"] or 0), int(metrics["num_updated_rows"] or 0)
    merge_commit = (
        DeltaTable.forName(spark, target_fq).history(10)
        .where((F.col("operation") == "MERGE") & (F.col("version") > version_before))
        .orderBy("version").select("operationMetrics").first()
    )
    if merge_commit is None:          # MERGE không tạo commit (không có thay đổi)
        return 0, 0
    om = merge_commit["operationMetrics"] or {}
    return int(om.get("numTargetRowsInserted", 0)), int(om.get("numTargetRowsUpdated", 0))


def append_rows(table: str, rows: list) -> None:
    """Append danh sách dict vào bảng Delta theo đúng schema bảng (thứ tự + kiểu cột);
    key thiếu trong dict -> NULL. Có thử lại khi xung đột ghi.

    Input : table — tên đầy đủ ; rows — list[dict], key = tên cột
    Output: không
    Ví dụ : append_rows("lh_vv_ctrl.dbo.ctrl_log_table_run",
                        [{"exec_id": "826b...", "trg_tbl": "slv_products", "status": "SUCCESS", ...}])
    """
    if not rows:
        return
    schema = spark.table(table).schema
    df = spark.createDataFrame([tuple(r.get(f.name) for f in schema.fields) for r in rows], schema)
    with_retry(lambda: df.write.format("delta").mode("append").saveAsTable(table), table)


class DeltaReadUnsupported(Exception):
    """Không đọc theo version được: log đã bị dọn, file đã bị VACUUM, bảng không còn append-only,
    hoặc bảng dùng tính năng cần Delta reader (deletion vector, column mapping, partition).
    read_incremental: dừng run (VersionGapError), hoặc đọc FULL nếu allow_full_scan."""


class VersionGapError(Exception):
    """Không xác định được đủ các commit raw cần đọc (chưa có last_src_version, log/file đã bị dọn,
    raw bị sửa/xoá hoặc bị tạo lại). Dừng run để không tiến watermark qua dữ liệu chưa đọc.
    Xử lý: chạy lại 1 lần với allow_full_scan = True (đọc toàn bộ raw, state lọc event đã áp dụng)."""


# Chỉ lấy các trường cần trong 1 dòng action của _delta_log/<version>.json (Delta protocol)
DELTA_LOG_SCHEMA = StructType([
    StructField("add", StructType([
        StructField("path", StringType()),
        StructField("size", LongType()),
        StructField("dataChange", BooleanType()),
        StructField("deletionVector", StructType([StructField("storageType", StringType())])),
    ])),
    StructField("remove", StructType([
        StructField("path", StringType()),
        StructField("dataChange", BooleanType()),
    ])),
    StructField("metaData", StructType([
        StructField("id", StringType()),
        StructField("partitionColumns", ArrayType(StringType())),
        StructField("configuration", MapType(StringType(), StringType())),
    ])),
    StructField("protocol", StructType([StructField("minReaderVersion", IntegerType())])),
])


def table_detail(table: str) -> dict:
    """Thông tin bảng Delta từ DESCRIBE DETAIL.

    Input : table — tên đầy đủ, vd "lh_vv_bronze.dbo.partner_raw_data"
    Output: dict, các khoá hay dùng: location, partitionColumns, minReaderVersion, numFiles, properties
    Ví dụ : table_detail("lh_vv_bronze.dbo.partner_raw_data")["location"]
            -> "abfss://68c8...@onelake.dfs.fabric.microsoft.com/8edc.../Tables/dbo/partner_raw_data"
    """
    return spark.sql(f"DESCRIBE DETAIL {quote_name(table)}").first().asDict()


def delta_added_files(detail: dict, v_from: int, v_to: int) -> list:
    """File dữ liệu được thêm (add, dataChange = true) trong các commit (v_from, v_to], đọc thẳng
    _delta_log/<version 20 chữ số>.json. Commit bảo trì (OPTIMIZE, auto compaction: dataChange = false) bị bỏ qua.
    Chỉ dùng cho bảng append-only; trường hợp đọc file trực tiếp không còn đúng -> DeltaReadUnsupported:
      - thiếu file log trong khoảng (log quá delta.logRetentionDuration đã bị dọn)
      - có remove dataChange = true (DELETE/UPDATE/MERGE/RESTORE trên bảng)
      - có deletion vector, protocol nâng reader version, bật column mapping, bảng có partition

    Input : detail — kết quả table_detail ; v_from — version đã xử lý ; v_to — version đã chốt
    Output: list[(path tương đối, size byte)] theo thứ tự commit; [] nếu v_to <= v_from
    Ví dụ : delta_added_files(detail, 18540, 18543)
            -> [("part-00000-1a2b...-c000.snappy.parquet", 11314), ("part-00000-3c4d...parquet", 9876), ...]
    """
    if v_to <= v_from:
        return []
    if detail.get("partitionColumns"):
        raise DeltaReadUnsupported(f"bảng có partition {detail['partitionColumns']}")
    if (detail.get("minReaderVersion") or 1) > 1:
        raise DeltaReadUnsupported(f"minReaderVersion = {detail['minReaderVersion']} (có thể dùng deletion vector / column mapping)")
    location = detail["location"].rstrip("/")
    log_paths = [f"{location}/_delta_log/{v:020d}.json" for v in range(v_from + 1, v_to + 1)]
    try:
        actions = spark.read.schema(DELTA_LOG_SCHEMA).json(log_paths)
    except AnalysisException as e:
        raise DeltaReadUnsupported(
            f"thiếu file _delta_log trong khoảng v{v_from + 1}..v{v_to} (đã bị dọn?): {short_error(e, 300)}") from e
    rows = (
        actions.where("add IS NOT NULL OR remove IS NOT NULL OR metaData IS NOT NULL OR protocol IS NOT NULL")
        .select(
            F.input_file_name().alias("log_file"),
            F.col("add.path").alias("path"), F.col("add.size").alias("size"),
            F.col("add.dataChange").alias("data_change"),
            F.col("add.deletionVector").isNotNull().alias("has_dv"),
            F.col("remove.dataChange").alias("remove_data_change"),
            F.col("metaData.partitionColumns").alias("partition_columns"),
            F.col("metaData.configuration").alias("configuration"),
            F.col("protocol.minReaderVersion").alias("reader_version"),
        )
        .collect()
    )
    files = []
    for r in sorted(rows, key=lambda x: x["log_file"]):          # tên file log = version đệm 0 -> đúng thứ tự commit
        version = r["log_file"].rsplit("/", 1)[-1][:20]
        if r["remove_data_change"]:
            raise DeltaReadUnsupported(f"v{int(version)} xoá/sửa dữ liệu (remove dataChange=true): bảng không còn append-only")
        if r["has_dv"]:
            raise DeltaReadUnsupported(f"v{int(version)} có deletion vector")
        if (r["reader_version"] or 1) > 1:
            raise DeltaReadUnsupported(f"v{int(version)} nâng minReaderVersion = {r['reader_version']}")
        if r["partition_columns"]:
            raise DeltaReadUnsupported(f"v{int(version)} đổi partition {r['partition_columns']}")
        if (r["configuration"] or {}).get("delta.columnMapping.mode", "none") != "none":
            raise DeltaReadUnsupported(f"v{int(version)} bật column mapping")
        if r["path"] is not None and r["data_change"]:
            files.append((r["path"], int(r["size"] or 0)))
    return files


def read_delta_files(table: str, location: str, paths: list) -> DataFrame:
    """Đọc trực tiếp các file parquet của bảng Delta (không qua Delta reader) theo schema hiện tại của bảng.
    Chỉ đúng khi delta_added_files đã kiểm tra (không deletion vector, column mapping, partition).
    File thiếu cột mới thêm sau này -> cột đó NULL.

    Input : table — tên đầy đủ (lấy schema) ; location — table_detail()["location"] ; paths — path từ delta_added_files
    Output: DataFrame đủ cột của bảng
    Ví dụ : read_delta_files("lh_vv_bronze.dbo.partner_raw_data", loc, ["part-00000-1a2b...parquet"])
            -> DataFrame(before, after, source, op, ..., EventProcessedUtcTime, ...)
            File đã bị VACUUM -> DeltaReadUnsupported
    """
    base = location.rstrip("/")
    full = [p if "://" in p else f"{base}/{unquote(p)}" for p in paths]    # path trong log được URL-encode
    try:
        return spark.read.schema(spark.table(table).schema).parquet(*full)
    except AnalysisException as e:
        raise DeltaReadUnsupported(f"file dữ liệu không còn (đã VACUUM?): {short_error(e, 300)}") from e


## 11. run_control_utils — Điều khiển lần chạy qua bảng control

| Hàm | Mục đích |
|---|---|
| `load_watermark(ctx)` | Đọc dòng watermark của nguồn: `last_src_version` (con trỏ chính), `last_src_table_id`, `watermark_value` |
| `pin_snapshot(ctx)` | Chốt version + table id của raw; lần chạy chỉ đọc dữ liệu tới version này |
| `version_gap_reason(ctx)` | Lý do không đọc theo version được (chưa có version, bảng bị tạo lại, version lùi) hoặc None |
| `choose_read_mode(ctx)` | `VERSION` (mặc định) / `FULL` (`full_reload` hoặc `allow_full_scan`); có khoảng trống mà không cho phép FULL → `VersionGapError` |
| `read_incremental(ctx, columns, wm_alias)` | Đọc raw theo cách đã chọn; không bao giờ tự lọc theo thời gian. Ghi `ctx.read_mode`, `ctx.read_note` |
| `watermark_has(*cols)` | Bảng watermark đã có các cột thêm sau (`last_src_table_id`, `lock_exec_id`, `lock_at`) chưa |
| `acquire_lock(ctx, extra)` | Chống chạy chồng **nguyên tử**: UPDATE có điều kiện trên dòng watermark (`lock_exec_id` trống hoặc quá `running_timeout_minutes`) rồi đọc lại xác nhận. Chưa có cột khoá → `claim_run` |
| `release_lock(ctx)` | Nhả khoá trong `finally` (chỉ khi run này còn giữ) |
| `claim_run(ctx)` | Dự phòng khi chưa có cột khoá: có run RUNNING còn hạn thì trả False (không nguyên tử); RUNNING quá hạn → ABANDONED |
| `update_watermark(ctx, values, condition)` | Cập nhật dòng watermark (thêm điều kiện nếu có, luôn kèm `updated_at`) |
| `finalize_watermark(ctx, status, watermark_to, error)` | Cuối run: thành công thì tiến watermark + `last_src_version` + `last_success_at` — **chỉ tiến, không lùi**, chỉ khi còn giữ khoá; lỗi thì giữ nguyên, chỉ ghi trạng thái. Không ghi được → trả cảnh báo |
| `run_log_row(...)` / `write_run_log(row)` | Dựng và ghi 1 dòng `ctrl_log_run` theo `exec_id` (mở RUNNING, đóng cuối run) |
| `table_log_row(res, ctx)` | Dựng 1 dòng `ctrl_log_table_run` từ `TableResult` |
| `decide_status(results)` | Trạng thái run từ trạng thái các bảng |
| `build_summary(...)` | JSON tóm tắt lần chạy (exit value cho pipeline) |


In [ ]:
# =============================================================================
# 11. run_control_utils
# =============================================================================
def load_watermark(ctx: RunContext) -> None:
    """Đọc đúng 1 dòng watermark cấp nguồn (trg_tbl IS NULL) và điền vào ctx:
    watermark_id, watermark_column, watermark_from, src_version_from, src_table_id_from.
    src_version_from (last_src_version) là con trỏ đọc chính; last_src_table_id dùng phát hiện raw bị tạo lại
    (cột thêm 02/10 — bảng chưa có cột thì bỏ qua).

    Input : ctx — RunContext (src_schema, src_tbl)
    Output: không (cập nhật ctx). Không đúng 1 dòng hoặc watermark_column không hợp lệ -> ConfigError
    Ví dụ : -> ctx.src_version_from = 18543, ctx.src_table_id_from = "65f3fbf1-...", ctx.watermark_column = "EventProcessedUtcTime"
    """
    has_tid = watermark_has("last_src_table_id")
    rows = spark.sql(f"""
        SELECT watermark_id, watermark_column, watermark_value, last_src_version,
               {"last_src_table_id" if has_tid else "CAST(NULL AS STRING)"} AS last_src_table_id
        FROM {T_WATERMARK}
        WHERE src_schema = :ss AND src_tbl = :st AND trg_tbl IS NULL
    """, args={"ss": ctx.src_schema, "st": ctx.src_tbl}).collect()
    if len(rows) != 1:
        raise ConfigError(f"ctrl_mng_watermark cần đúng 1 dòng cấp nguồn cho {ctx.raw_fq}, đang có {len(rows)}")
    wm = rows[0]
    if not is_identifier(wm["watermark_column"]):
        raise ConfigError(f"watermark_column không hợp lệ: {wm['watermark_column']}")
    ctx.watermark_id = wm["watermark_id"]
    ctx.watermark_column = wm["watermark_column"]
    ctx.watermark_from = wm["watermark_value"]
    ctx.src_version_from = wm["last_src_version"]
    ctx.src_table_id_from = wm["last_src_table_id"]


def pin_snapshot(ctx: RunContext) -> None:
    """Chốt version hiện tại + Delta table id của bảng raw vào ctx.
    Dữ liệu ghi vào raw trong lúc chạy không lẫn vào lần này; version này cũng là last_src_version
    để pre-check Get Metadata biết lần sau có commit mới hay không.

    Input : ctx — RunContext
    Output: không (ctx.src_version_to, ctx.src_table_id, ctx.src_detail)
    Ví dụ : pin_snapshot(ctx) -> ctx.src_version_to = 18543, ctx.src_table_id = "65f3fbf1-a67f-4580-9e87-00cd798a2196"
    """
    ctx.src_version_to = current_version(ctx.raw_fq)
    ctx.src_detail = table_detail(ctx.raw_fq)
    ctx.src_table_id = ctx.src_detail.get("id")


def version_gap_reason(ctx: RunContext) -> Optional[str]:
    """Lý do không thể đọc theo version (chỉ xét thông tin watermark, chưa đọc _delta_log). None = đọc VERSION được.

    Input : ctx — RunContext đã qua load_watermark + pin_snapshot
    Output: str | None
    Ví dụ : last_src_version NULL -> "chưa có last_src_version (lần đầu / sau reset / sau cutover)"
            table id lưu "a1..." khác hiện tại "b2..." -> "bảng raw đã bị tạo lại ..."
            last_src_version 18600, version chốt 18543 -> "version raw 18543 nhỏ hơn last_src_version 18600 ..."
    """
    if ctx.src_version_from is None:
        return "chưa có last_src_version (lần đầu / sau reset / sau cutover)"
    if ctx.src_table_id_from and ctx.src_table_id and ctx.src_table_id_from != ctx.src_table_id:
        return f"bảng raw đã bị tạo lại (table id {ctx.src_table_id_from} -> {ctx.src_table_id})"
    if ctx.src_version_to < ctx.src_version_from:
        return f"version raw {ctx.src_version_to} nhỏ hơn last_src_version {ctx.src_version_from} (bảng raw bị tạo lại?)"
    return None


def _gap(ctx: RunContext, reason: str) -> tuple:
    """Có khoảng trống: cho phép FULL thì trả ("FULL", lý do), không thì raise VersionGapError."""
    if ctx.allow_full_scan:
        return "FULL", f"{reason} | allow_full_scan"
    raise VersionGapError(f"{reason}. Không tiến watermark khi chưa đọc đủ dữ liệu: chạy lại 1 lần với "
                          f"allow_full_scan = True (đọc toàn bộ raw, state lọc event đã áp dụng).")


def choose_read_mode(ctx: RunContext) -> tuple:
    """Chọn cách đọc raw cho lần chạy (gọi sau load_watermark + pin_snapshot). Không có kiểu lọc theo thời gian:
    lọc theo EventProcessedUtcTime có thể bỏ sót event commit trễ rồi vẫn tiến watermark qua chúng.
    - VERSION: đọc file thêm trong các commit (last_src_version, version đã chốt]
    - FULL   : full_reload, hoặc có khoảng trống (version_gap_reason) và allow_full_scan = True
    - Có khoảng trống, allow_full_scan = False -> VersionGapError (run FAILED, watermark giữ nguyên)

    Input : ctx — RunContext (full_reload, allow_full_scan, src_version_from/to, src_table_id_from/src_table_id)
    Output: (mode, lý do hoặc None)
    Ví dụ : last_src_version 18500, version chốt 18543 -> ("VERSION", None)
            last_src_version NULL, allow_full_scan False -> VersionGapError
            last_src_version NULL, allow_full_scan True  -> ("FULL", "chưa có last_src_version ... | allow_full_scan")
    """
    if ctx.full_reload:
        return "FULL", "full_reload"
    reason = version_gap_reason(ctx)
    if reason:
        return _gap(ctx, reason)
    return "VERSION", None


def read_incremental(ctx: RunContext, columns: list, wm_alias: str = "_wm") -> DataFrame:
    """Đọc phần raw cần xử lý của lần chạy, trả các cột `columns` + cột watermark (ép TIMESTAMP, tên wm_alias).
    Cách đọc theo choose_read_mode. VERSION gặp DeltaReadUnsupported (thiếu log, file đã VACUUM, raw bị sửa/xoá…):
    allow_full_scan -> đọc FULL (log WARN); không -> VersionGapError. Ghi ctx.read_mode, ctx.read_note (ctrl_log_run).

    | Cách    | Đọc gì                                                       | Chi phí                  |
    |---------|--------------------------------------------------------------|--------------------------|
    | VERSION | File add (dataChange) trong commit (last_src_version, v_to]  | Theo lượng dữ liệu mới   |
    | FULL    | VERSION AS OF v_to, toàn bộ (state vẫn lọc event đã áp dụng) | Quét cả bảng             |

    Input : ctx — RunContext đã qua load_watermark + pin_snapshot ; columns — cột cần lấy ; wm_alias — tên cột watermark
    Output: DataFrame(columns..., wm_alias)
    Ví dụ : read_incremental(ctx, ["before", "after", "source", "op"], "_raw_ingested_at")
            -> ctx.read_mode = "VERSION", ctx.read_note = "v18501..v18543: 43 commit, 41 file, 6.2 MB"
    """
    def project(df: DataFrame) -> DataFrame:
        return df.select(*[F.col(c) for c in columns],
                         F.col(ctx.watermark_column).cast("timestamp").alias(wm_alias))

    mode, note = choose_read_mode(ctx)
    if mode == "VERSION":
        try:
            detail = ctx.src_detail or table_detail(ctx.raw_fq)
            files = delta_added_files(detail, ctx.src_version_from, ctx.src_version_to)
            note = (f"v{ctx.src_version_from + 1}..v{ctx.src_version_to}: "
                    f"{ctx.src_version_to - ctx.src_version_from} commit, {len(files)} file, "
                    f"{sum(sz for _, sz in files) / 1e6:.1f} MB"
                    if ctx.src_version_to > ctx.src_version_from
                    else f"không có commit mới (v{ctx.src_version_to})")
            if files:
                df = project(read_delta_files(ctx.raw_fq, detail["location"], [p for p, _ in files]))
            else:
                df = spark.createDataFrame([], project(spark.table(ctx.raw_fq)).schema)
            ctx.read_mode, ctx.read_note = "VERSION", note       # chỉ ghi khi đọc được (lỗi -> run log không ghi nhầm VERSION)
            return df
        except DeltaReadUnsupported as e:
            mode, note = _gap(ctx, f"không đọc theo version được: {e}")      # không cho phép FULL -> raise
            log(f"đọc FULL: {note}", "WARN")

    ctx.read_mode, ctx.read_note = mode, note
    return project(spark.sql(f"SELECT * FROM {quote_name(ctx.raw_fq)} VERSION AS OF {int(ctx.src_version_to)}"))


LOCK_COLUMNS = ("lock_exec_id", "lock_at")


def watermark_has(*cols: str) -> bool:
    """ctrl_mng_watermark đã có đủ các cột này chưa (cột thêm dần bằng ALTER TABLE: last_src_table_id 02/10, khoá 03/10).

    Input : cols — tên cột
    Output: bool
    Ví dụ : watermark_has("lock_exec_id", "lock_at") -> False khi chưa chạy ALTER TABLE ... ADD COLUMNS
    """
    existing = set(spark.table(T_WATERMARK).columns)
    return all(c in existing for c in cols)


def _abandon_stale_runs(ctx: RunContext) -> list:
    """Run log RUNNING của nguồn (khác run này) quá running_timeout_minutes -> ABANDONED (notebook đã chết mà không
    đóng log). Trả exec_id RUNNING còn trong hạn.

    Input : ctx — RunContext (exec_id, src_schema, src_tbl, running_timeout_minutes)
    Output: list[str]
    Ví dụ : 1 run RUNNING bắt đầu 3 giờ trước (timeout 60) -> run đó thành ABANDONED, trả []
    """
    rows = spark.sql(f"""
        SELECT exec_id, started_at FROM {T_LOG_RUN}
        WHERE src_schema = :ss AND src_tbl = :st AND status = 'RUNNING' AND exec_id <> :me
    """, args={"ss": ctx.src_schema, "st": ctx.src_tbl, "me": ctx.exec_id}).collect()
    cutoff = utc_now() - timedelta(minutes=ctx.running_timeout_minutes)
    abandoned = [r["exec_id"] for r in rows if r["started_at"] is None or r["started_at"] < cutoff]
    if abandoned:
        with_retry(lambda: DeltaTable.forName(spark, T_LOG_RUN).update(
            F.col("exec_id").isin(abandoned),
            {
                "status": F.lit("ABANDONED"),
                "ended_at": F.current_timestamp(),
                "error_message": F.lit(f"RUNNING quá {ctx.running_timeout_minutes} phút, coi như đã dừng"),
            },
        ), "ctrl_log_run")
        log(f"Đánh dấu ABANDONED: {abandoned}", "WARN")
    return [r["exec_id"] for r in rows if r["exec_id"] not in abandoned]


def claim_run(ctx: RunContext) -> bool:
    """Chống chạy chồng bằng run log — chỉ dùng khi ctrl_mng_watermark chưa có cột khoá (acquire_lock tự chuyển sang).
    Không nguyên tử: 2 run khởi động cùng lúc đều có thể thấy "không có run RUNNING" và cùng chạy.

    Input : ctx — RunContext (src_schema, src_tbl, running_timeout_minutes)
    Output: bool — True = được chạy
    Ví dụ : có run RUNNING bắt đầu 5 phút trước -> False
    """
    active = _abandon_stale_runs(ctx)
    if active:
        log(f"Đang có lần chạy khác: {active} -> bỏ qua", "WARN")
        return False
    return True


def update_watermark(ctx: RunContext, values: dict, condition: Optional[Column] = None) -> None:
    """Cập nhật dòng watermark của nguồn (thêm điều kiện nếu có); luôn ghi thêm updated_at = now.
    Lỗi xung đột ghi -> with_retry chạy lại; điều kiện được xét lại trên snapshot mới nên thử lại vẫn an toàn.

    Input : ctx — RunContext (watermark_id) ; values — {tên cột: Column} ; condition — Column bool hoặc None
    Output: không (không biết số dòng đã sửa -> hàm gọi tự đọc lại nếu cần)
    Ví dụ : update_watermark(ctx, {"status": F.lit("FAILED")}, F.col("lock_exec_id") == F.lit(ctx.exec_id))
    """
    cond = F.col("watermark_id") == F.lit(ctx.watermark_id)
    if condition is not None:
        cond = cond & condition
    with_retry(lambda: DeltaTable.forName(spark, T_WATERMARK).update(
        cond, {**values, "updated_at": F.current_timestamp()}), "ctrl_mng_watermark")


def _lock_expired_sql(ctx: RunContext) -> str:
    """Biểu thức SQL: khoá đã quá running_timeout_minutes (lock_at NULL coi như quá hạn). Tính bằng current_timestamp()
    giống lúc ghi lock_at nên không phụ thuộc múi giờ session."""
    return f"coalesce(lock_at < current_timestamp() - INTERVAL {int(ctx.running_timeout_minutes)} MINUTES, true)"


def read_lock(ctx: RunContext):
    """Khoá hiện tại trên dòng watermark: Row(lock_exec_id, lock_at, expired)."""
    return spark.sql(f"""
        SELECT lock_exec_id, lock_at, {_lock_expired_sql(ctx)} AS expired
        FROM {T_WATERMARK} WHERE watermark_id = :wid
    """, args={"wid": ctx.watermark_id}).first()


def acquire_lock(ctx: RunContext, extra: Optional[dict] = None) -> tuple:
    """Nhận khoá chạy của nguồn trên chính dòng watermark (nguyên tử nhờ Delta optimistic concurrency):
      UPDATE SET lock_exec_id = <exec_id>, lock_at = now [, extra]
      WHERE watermark_id = … AND (lock_exec_id IS NULL OR lock_at quá running_timeout_minutes)
    rồi đọc lại: lock_exec_id = exec_id của run này mới là nhận được. 2 run cùng UPDATE thì 1 run bị xung đột ghi,
    thử lại thấy khoá đã có chủ -> không sửa gì -> đọc lại thấy exec_id khác -> không chạy.
    Khoá quá hạn (run trước chết không nhả) được lấy lại; run log RUNNING quá hạn -> ABANDONED.
    Bảng chưa có cột lock_exec_id/lock_at -> WARN, dùng claim_run (run log, không nguyên tử).

    Input : ctx — RunContext (watermark_id, exec_id, running_timeout_minutes) ;
            extra — cột ghi cùng lúc khi nhận được khoá, vd {"status": F.lit("RUNNING"), "last_run_id": ...}
    Output: (True, None) | (False, "ai đang giữ khoá"). Ghi ctx.lock_mode = WATERMARK | RUN_LOG
    Ví dụ : khoá trống -> (True, None)
            exec_id "9f1c..." giữ từ 08:00, timeout 60, bây giờ 08:20 -> (False, "exec_id 9f1c... đang giữ khoá từ ...")
    """
    if not watermark_has(*LOCK_COLUMNS):
        ctx.lock_mode = "RUN_LOG"
        log(f"{T_WATERMARK} chưa có cột {', '.join(LOCK_COLUMNS)} -> chống chạy chồng bằng run log (không nguyên tử). "
            f"Thêm cột: ALTER TABLE {T_WATERMARK} ADD COLUMNS (lock_exec_id STRING, lock_at TIMESTAMP)", "WARN")
        if not claim_run(ctx):
            return False, "đang có run RUNNING khác trong ctrl_log_run"
        if extra:
            update_watermark(ctx, extra)
        return True, None

    ctx.lock_mode = "WATERMARK"
    before = read_lock(ctx)
    if before["lock_exec_id"] and not before["expired"]:
        return False, f"exec_id {before['lock_exec_id']} đang giữ khoá từ {before['lock_at']}"
    free = F.col("lock_exec_id").isNull() | F.expr(_lock_expired_sql(ctx))
    update_watermark(ctx, {"lock_exec_id": F.lit(ctx.exec_id), "lock_at": F.current_timestamp(), **(extra or {})}, free)
    after = read_lock(ctx)
    if after["lock_exec_id"] != ctx.exec_id:
        return False, f"exec_id {after['lock_exec_id']} nhận khoá trước (từ {after['lock_at']})"
    if before["lock_exec_id"]:
        log(f"Lấy lại khoá quá hạn của exec_id {before['lock_exec_id']} (giữ từ {before['lock_at']})", "WARN")
    active = _abandon_stale_runs(ctx)
    if active:
        log(f"Run log còn RUNNING trong hạn nhưng không giữ khoá: {active}", "WARN")
    return True, None


def release_lock(ctx: RunContext) -> None:
    """Nhả khoá (chỉ khi run này đang giữ: WHERE lock_exec_id = exec_id). Gọi trong finally, sau khi đóng run log.
    Khoá đã bị run khác lấy lại (quá hạn) -> không sửa gì.

    Input : ctx — RunContext (lock_mode, exec_id)
    Output: không
    """
    if ctx.lock_mode != "WATERMARK":
        return
    update_watermark(ctx, {"lock_exec_id": F.lit(None).cast("string"), "lock_at": F.lit(None).cast("timestamp")},
                     F.col("lock_exec_id") == F.lit(ctx.exec_id))


def finalize_watermark(ctx: RunContext, status: str, watermark_to: Optional[datetime],
                       error: Optional[str]) -> Optional[str]:
    """Cập nhật watermark cuối run. Chỉ khi ctx.updates_watermark (incremental đầy đủ). Có khoá (WATERMARK) thì chỉ ghi
    khi run này còn giữ khoá.
    - SUCCESS / NO_DATA: watermark_value = max(cũ, watermark_to), last_src_version = version đã chốt, last_success_at,
      status SUCCESS. Chỉ tiến, không lùi: ghi khi last_src_version NULL / ≤ version chốt / raw bị tạo lại
      (last_src_table_id NULL hoặc khác). Ghi cùng lúc vì pre-check dùng cả last_src_version và last_success_at.
    - Trạng thái khác: giữ nguyên watermark, chỉ ghi status FAILED + lỗi -> lần sau đọc lại cùng khoảng.
    Đọc lại dòng sau khi ghi; không ghi được (mất khoá, run khác đã tiến xa hơn) -> trả cảnh báo.

    Input : ctx — RunContext ; status — trạng thái run ; watermark_to — max thời điểm ingest đã đọc ; error — lỗi
    Output: None | cảnh báo (str) — hàm gọi ghi vào error_message của run log
    Ví dụ : status "SUCCESS", version chốt 120, dòng đang 110 -> last_src_version = 120, trả None
            status "SUCCESS", version chốt 110, dòng đang 120 -> không ghi, trả "không tiến watermark: ..."
    """
    if not ctx.updates_watermark:
        return None
    own = (F.col("lock_exec_id") == F.lit(ctx.exec_id)) if ctx.lock_mode == "WATERMARK" else F.lit(True)
    success = status in ("SUCCESS", "NO_DATA")
    if success:
        values = {
            "status": F.lit("SUCCESS"),
            "last_success_at": F.current_timestamp(),
            "last_src_version": F.lit(ctx.src_version_to).cast("bigint"),
            "last_run_id": F.lit(ctx.run_id),
            "flow_name": F.lit(ctx.nb_name),
            "error_message": F.lit(None).cast("string"),
        }
        forward = F.col("last_src_version").isNull() | (F.col("last_src_version") <= F.lit(ctx.src_version_to))
        if watermark_has("last_src_table_id"):                           # cột thêm 02/10
            values["last_src_table_id"] = F.lit(ctx.src_table_id).cast("string")
            forward = (forward | F.col("last_src_table_id").isNull()
                       | (F.col("last_src_table_id") != F.lit(ctx.src_table_id)))
        candidates = [t for t in (ctx.watermark_from, watermark_to) if t is not None]
        if candidates:
            values["watermark_value"] = F.lit(max(candidates)).cast("timestamp")
        update_watermark(ctx, values, own & forward)
    else:
        update_watermark(ctx, {
            "status": F.lit("FAILED"),
            "last_run_id": F.lit(ctx.run_id),
            "error_message": F.lit(error).cast("string"),
        }, own)

    lock_col = "lock_exec_id" if ctx.lock_mode == "WATERMARK" else "CAST(NULL AS STRING)"
    row = spark.sql(f"""
        SELECT status, last_run_id, last_src_version, {lock_col} AS lock_exec_id
        FROM {T_WATERMARK} WHERE watermark_id = :wid
    """, args={"wid": ctx.watermark_id}).first()
    if ctx.lock_mode == "WATERMARK" and row["lock_exec_id"] != ctx.exec_id:
        note = (f"không ghi watermark: run đã mất khoá (lock_exec_id hiện tại {row['lock_exec_id']}; "
                f"run chạy quá running_timeout_minutes?)")
    elif success and not (row["status"] == "SUCCESS" and row["last_run_id"] == ctx.run_id
                          and row["last_src_version"] == ctx.src_version_to):
        note = (f"không tiến watermark: last_src_version hiện tại {row['last_src_version']} > version run này "
                f"{ctx.src_version_to} (run khác đã xử lý tới version mới hơn)")
    else:
        return None
    log(note, "WARN")
    return note


def run_log_row(ctx: RunContext, status: str, profile: Optional[Profile] = None, results: Optional[list] = None,
                error: Optional[str] = None, ended_at: Optional[datetime] = None) -> dict:
    """Dựng 1 dòng ctrl_log_run (đủ cột) từ trạng thái hiện tại của lần chạy.

    Input : ctx ; status — RUNNING | SUCCESS | PARTIAL_FAILED | FAILED | NO_DATA | SKIPPED_CONCURRENT
            profile — Profile (None khi chưa đọc raw) ; results — list[TableResult] ; error ; ended_at
    Output: dict {tên cột: giá trị}
    Ví dụ : run_log_row(ctx, "RUNNING") -> {"exec_id": "826b...", "status": "RUNNING", "read_rows": None, ...}
    """
    results = results or []
    counts = Counter(r.status for r in results)
    return {
        "exec_id": ctx.exec_id, "run_id": ctx.run_id, "pl_name": ctx.pl_name, "nb_name": ctx.nb_name,
        "src_schema": ctx.src_schema, "src_tbl": ctx.src_tbl, "watermark_id": ctx.watermark_id,
        "run_mode": ctx.run_mode, "table_filter": ",".join(ctx.table_filter) or None,
        "src_version_from": ctx.src_version_from, "src_version_to": ctx.src_version_to,
        "watermark_from": ctx.watermark_from, "watermark_to": profile.watermark_to if profile else None,
        "overlap_minutes": ctx.overlap_minutes,
        "read_mode": ctx.read_mode, "read_note": ctx.read_note,      # ghi nếu bảng có cột (ALTER TABLE 02/10)
        "read_rows": profile.read_rows if profile else None,
        "valid_rows": profile.valid_rows if profile else None,
        "ignored_rows": profile.ignored_rows if profile else None,
        "rejected_rows": (profile.rejected_rows + sum(r.rejected_rows for r in results if r.status == "SUCCESS"))
                         if profile else None,
        "ignored_detail": json.dumps(profile.ignored_detail, ensure_ascii=False)
                          if profile and profile.ignored_detail else None,
        "tbl_total_count": len(results) if results else None,
        "tbl_success_count": counts["SUCCESS"] if results else None,
        "tbl_failed_count": counts["FAILED"] + counts["SKIPPED"] if results else None,
        "tbl_no_data_count": counts["NO_DATA"] if results else None,
        "status": status, "started_at": ctx.started_at, "ended_at": ended_at,
        "duration_ms": int((ended_at - ctx.started_at).total_seconds() * 1000) if ended_at else None,
        "run_params": ctx.run_params, "error_message": error,
    }


def write_run_log(row: dict) -> None:
    """Ghi dòng run log bằng MERGE theo exec_id: lần đầu insert (RUNNING), lần sau update cùng dòng.

    Input : row — dict từ run_log_row
    Output: không
    Ví dụ : write_run_log(run_log_row(ctx, "RUNNING")) ... write_run_log(run_log_row(ctx, "SUCCESS", ...))
            -> ctrl_log_run chỉ có 1 dòng cho exec_id này, status cuối = SUCCESS
    """
    schema = spark.table(T_LOG_RUN).schema
    df = spark.createDataFrame([tuple(row.get(f.name) for f in schema.fields)], schema)
    with_retry(lambda: (
        DeltaTable.forName(spark, T_LOG_RUN).alias("t")
        .merge(df.alias("s"), "t.exec_id = s.exec_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    ), "ctrl_log_run")


def table_log_row(res: TableResult, ctx: RunContext) -> dict:
    """Dựng 1 dòng ctrl_log_table_run từ kết quả 1 bảng. Bảng thành công có NULL do chuyển kiểu
    thì ghi chi tiết theo cột vào error_message dạng JSON.

    Input : res — TableResult ; ctx — RunContext
    Output: dict {tên cột: giá trị}
    Ví dụ : -> {"exec_id": "826b...", "trg_tbl": "slv_orders", "status": "SUCCESS", "inserted_rows": 2,
                "cast_null_rows": 82, "error_message": '{"cast_null": {"total_payment": 82}}', ...}
    """
    return {
        "exec_id": ctx.exec_id, "run_id": ctx.run_id, "pl_name": ctx.pl_name,
        "src_schema": ctx.src_schema, "src_tbl": ctx.src_tbl, "src_object": res.spec.src_object,
        "trg_schema": res.spec.trg_schema, "trg_tbl": res.spec.trg_tbl,
        "priority": res.spec.priority, "attempt_no": res.attempt_no,
        "input_rows": res.input_rows, "stale_rows": res.stale_rows, "applicable_rows": res.applicable_rows,
        "entity_rows": res.entity_rows, "upsert_event_rows": res.upsert_event_rows,
        "delete_event_rows": res.delete_event_rows, "inserted_rows": res.inserted_rows,
        "updated_rows": res.updated_rows, "cast_null_rows": res.cast_null_rows,
        "rejected_rows": res.rejected_rows, "status": res.status,
        "started_at": res.started_at, "ended_at": res.ended_at, "duration_ms": res.duration_ms,
        "error_message": res.error_message
                         or (json.dumps({"cast_null": res.cast_null_detail}) if res.cast_null_detail else None),
    }


def decide_status(results: list) -> str:
    """Trạng thái lần chạy từ trạng thái các bảng.

    Input : results — list[TableResult]
    Output: FAILED | PARTIAL_FAILED | NO_DATA | SUCCESS
    Ví dụ : [SUCCESS, NO_DATA] -> SUCCESS ; [SUCCESS, FAILED] -> PARTIAL_FAILED
            [FAILED, SKIPPED, NO_DATA] -> FAILED ; [NO_DATA] hoặc [] -> NO_DATA
    """
    processed = [r for r in results if r.status != "NO_DATA"]
    failed = [r for r in results if r.status in ("FAILED", "SKIPPED")]
    if failed:
        return "FAILED" if len(failed) == len(processed) else "PARTIAL_FAILED"
    return "SUCCESS" if processed else "NO_DATA"


def build_summary(ctx: RunContext, status: str, profile: Optional[Profile], results: list,
                  error: Optional[str]) -> dict:
    """JSON tóm tắt lần chạy, dùng làm exit value cho pipeline.

    Input : ctx ; status ; profile ; results — list[TableResult] ; error
    Output: dict
    Ví dụ : {"status": "SUCCESS", "exec_id": "...", "src_version": 18543,
             "tables": {"total": 23, "ok": 14, "failed": 0, "skipped": 0, "no_data": 9},
             "rows": {"read": 1588, "valid": 1568, "inserted": 664, ...}, "duration_s": 312.5, ...}
    """
    counts = Counter(r.status for r in results)
    ok = [r for r in results if r.status == "SUCCESS"]
    return {
        "status": status, "exec_id": ctx.exec_id, "run_id": ctx.run_id, "run_mode": ctx.run_mode,
        "src": ctx.raw_fq, "src_version_from": ctx.src_version_from, "src_version": ctx.src_version_to,
        "read_mode": ctx.read_mode, "read_note": ctx.read_note,
        "watermark_from": ctx.watermark_from, "watermark_to": profile.watermark_to if profile else None,
        "tables": {"total": len(results), "ok": counts["SUCCESS"], "failed": counts["FAILED"],
                   "skipped": counts["SKIPPED"], "no_data": counts["NO_DATA"]},
        "failed_tables": [r.spec.trg_tbl for r in results if r.status in ("FAILED", "SKIPPED")],
        "rows": {
            "read": profile.read_rows if profile else 0,
            "valid": profile.valid_rows if profile else 0,
            "ignored": profile.ignored_rows if profile else 0,
            "rejected": (profile.rejected_rows if profile else 0) + sum(r.rejected_rows for r in ok),
            "inserted": sum(r.inserted_rows for r in ok),
            "updated": sum(r.updated_rows for r in ok),
            "cast_null": sum(r.cast_null_rows for r in ok),
        },
        "duration_s": round((utc_now() - ctx.started_at).total_seconds(), 1),
        "error": error,
    }


## 12. parallel_utils — Chạy song song theo wave

| Hàm | Mục đích |
|---|---|
| `run_in_waves(items_by_priority, worker, max_parallel, results, ...)` | Các việc cùng priority chạy song song (thread pool); wave sau chạy khi wave trước xong; tuỳ chọn dừng các wave sau khi có việc lỗi |

Vì sao dùng thread trong 1 notebook: các việc dùng chung DataFrame đã cache trong cùng Spark session.
Mỗi luồng chỉ được ghi bảng của riêng nó; bảng dùng chung (control) do luồng chính ghi sau khi xong.


In [ ]:
# =============================================================================
# 12. parallel_utils
# =============================================================================
def run_in_waves(items_by_priority: dict, worker: Callable, max_parallel: int, results: list,
                 is_failed: Callable = lambda r: False, on_skip: Optional[Callable] = None,
                 stop_on_failure: bool = False, label: Callable = str) -> None:
    """Chạy các việc theo wave: priority nhỏ trước; trong 1 wave chạy song song tối đa max_parallel luồng.

    Input : items_by_priority — {priority: [item, ...]}
            worker(item, wave) -> result: xử lý 1 việc; nên tự bắt lỗi (không raise) để không làm hỏng cả wave
            max_parallel — số luồng tối đa trong 1 wave
            results — list nhận kết quả; truyền từ ngoài vào để vẫn giữ kết quả nếu có lỗi bất ngờ
            is_failed(result) -> bool: kết quả có tính là lỗi không (dùng cho stop_on_failure)
            on_skip(item, wave) -> result: tạo kết quả cho việc bị bỏ qua khi stop_on_failure
            stop_on_failure — True: wave có việc lỗi thì không chạy các wave sau
            label(item) -> str: tên việc để ghi log
    Output: không (kết quả thêm vào `results` theo thứ tự wave, trong wave theo thứ tự item)
    Ví dụ : out = []
            run_in_waves({1: ["a", "b"], 2: ["c"]}, lambda x, w: f"{x}@w{w}", 2, out)
            out -> ["a@w1", "b@w1", "c@w2"]   (a, b chạy song song; c chạy sau)
    """
    stop = False
    for priority in sorted(items_by_priority):
        batch = items_by_priority[priority]
        if stop:
            if on_skip is not None:
                results += [on_skip(item, priority) for item in batch]
            continue
        log(f"priority {priority}: {len(batch)} việc {[label(i) for i in batch]}", wave=priority)
        # inheritable_thread_target: luồng con kế thừa job group của cell -> Cancel cell dừng được cả luồng con
        run_one = inheritable_thread_target(lambda item: worker(item, priority))
        with ThreadPoolExecutor(max_workers=max(1, min(max_parallel, len(batch))),
                                thread_name_prefix=f"w{priority}") as pool:
            wave_results = list(pool.map(run_one, batch))
        results += wave_results
        if stop_on_failure and any(is_failed(r) for r in wave_results):
            log("có việc lỗi, dừng các wave sau (stop_on_failure)", "WARN", priority)
            stop = True


## 13. state_reject_utils — State theo entity và bảng reject

**[03/10 3P]** Chuyển nguyên từ `NB_EXTRACT_PARTNER_CDC_BRZ_TO_SLV` (mục B, C) để nguồn 3rd-party dùng chung; logic không đổi
(trừ `state_rows`: `is_deleted` thêm `coalesce(..., false)` cho nguồn không có op — partner luôn có op nên kết quả như cũ).

Hợp đồng cột "event" (notebook phải dựng đủ trước khi gọi): `_cdc_ts_ms` BIGINT, `_cdc_lsn` BIGINT, `_raw_ingested_at` TIMESTAMP,
`_is_delete` INT (0/1), `_op` STRING (NULL được), `_entity_key` STRING.

| Nguồn | `_cdc_ts_ms` | `_cdc_lsn` | `_raw_ingested_at` | `_is_delete` | `_op` | `_entity_key` |
|---|---|---|---|---|---|---|
| partner (CDC) | `source.ts_ms` | `source.lsn` | `EventProcessedUtcTime` | op = d | c/r/u/d | khoá bảng đích |
| 3rd-party (snapshot) | `crawled_at` (ms) | NULL | `crawled_at` | 0 | NULL | `poi_id` (mọi bảng) |

| Hàm / hằng số | Mục đích |
|---|---|
| `STATE_COLUMNS` | Cột của `ctrl_cdc_state` theo thứ tự MERGE |
| `event_order()` | Thứ tự event `(ts_ms, lsn, thời điểm ingest, là xoá)` để so với state |
| `load_state(specs)` | State các bảng trong phạm vi, đọc 1 lần cho mọi luồng |
| `state_rows(spec, latest, ctx)` | Dòng state mới cho các entity vừa áp dụng (`latest`: 1 dòng / `_entity_key`) |
| `state_order_sql(alias)`, `STATE_NOT_OLDER` | Điều kiện MERGE state: chỉ ghi đè khi event mới ≥ event đang lưu (state không lùi) |
| `src_partition_cond(src_tbl)` | Điều kiện `t.src_tbl = '<nguồn>'` thêm vào MERGE: 2 bảng này partition theo `src_tbl` (NB_CREATE_DDL 03/10 - 3P) nên MERGE của 2 nguồn chạy song song không xung đột |
| `commit_state(results, src_tbl)` | MERGE state của các bảng SUCCESS vào `ctrl_cdc_state` |
| `merge_rejects(parts, note, src_tbl)` | MERGE các DataFrame reject (đúng schema `ctrl_cdc_reject`) theo `event_hash`; event đã có → `reject_count + 1` |


In [ ]:
# =============================================================================
# 13. state_reject_utils  [03/10 3P: chuyển từ NB_EXTRACT_PARTNER_CDC_BRZ_TO_SLV]
# =============================================================================
STATE_COLUMNS = ["trg_schema", "trg_tbl", "entity_key", "src_schema", "src_tbl", "src_object", "cdc_op",
                 "cdc_ts_ms", "cdc_lsn", "raw_ingested_at", "is_deleted", "src_version", "last_run_id", "updated_at"]


def event_order() -> Column:
    """Thứ tự event: (_cdc_ts_ms, _cdc_lsn, _raw_ingested_at, _is_delete); NULL thay bằng -1 / MIN_TS.
    Cùng kiểu và tên trường với state (_st_order của load_state) để so sánh struct trực tiếp; trường "del":
    trùng thứ tự thì event xoá được coi là mới hơn (giống ORDER_ASC/DESC của partner).

    Input : không (dùng các cột event của DataFrame đang xử lý)
    Output: Column struct<ts, lsn, ing, del>
    Ví dụ : df.where(event_order() > F.col("_st_order"))  -> event mới hơn state
    """
    return F.struct(
        F.col("_cdc_ts_ms").alias("ts"),
        F.coalesce(F.col("_cdc_lsn"), F.lit(-1).cast("bigint")).alias("lsn"),
        F.coalesce(F.col("_raw_ingested_at"), F.lit(MIN_TS).cast("timestamp")).alias("ing"),
        F.col("_is_delete").cast("int").alias("del"),
    )


def load_state(specs: list) -> DataFrame:
    """State (entity_key + thứ tự event đã áp dụng) của các bảng trong phạm vi chạy.

    Input : specs — list[TableSpec]
    Output: DataFrame(trg_schema, trg_tbl, _st_key, _st_order) — lazy, hàm gọi tự persist
    Ví dụ : load_state(specs).where("trg_tbl = 'slv_pn_products'")
    """
    targets = [f"{s.trg_schema}|{s.trg_tbl}" for s in specs]
    return (
        spark.table(T_CDC_STATE)
        .where(F.concat_ws("|", "trg_schema", "trg_tbl").isin(targets))
        .select(
            "trg_schema", "trg_tbl",
            F.col("entity_key").alias("_st_key"),
            F.struct(
                F.coalesce("cdc_ts_ms", F.lit(-1).cast("bigint")).alias("ts"),
                F.coalesce("cdc_lsn", F.lit(-1).cast("bigint")).alias("lsn"),
                F.coalesce("raw_ingested_at", F.lit(MIN_TS).cast("timestamp")).alias("ing"),
                F.coalesce(F.col("is_deleted").cast("int"), F.lit(0)).alias("del"),
            ).alias("_st_order"),
        )
    )


def state_rows(spec: TableSpec, latest: DataFrame, ctx: RunContext) -> DataFrame:
    """Dòng ctrl_cdc_state mới cho các entity vừa áp dụng (lazy, chỉ commit nếu bảng SUCCESS).
    [03/10 3P] is_deleted = coalesce(_op = 'd', false): nguồn snapshot có _op NULL -> false.

    Input : spec ; latest — DataFrame 1 dòng / _entity_key, có cột event ; ctx
    Output: DataFrame theo STATE_COLUMNS
    """
    return latest.select(
        F.lit(spec.trg_schema).alias("trg_schema"),
        F.lit(spec.trg_tbl).alias("trg_tbl"),
        F.col("_entity_key").alias("entity_key"),
        F.lit(ctx.src_schema).alias("src_schema"),
        F.lit(ctx.src_tbl).alias("src_tbl"),
        F.lit(spec.src_object).alias("src_object"),
        F.col("_op").alias("cdc_op"),
        F.col("_cdc_ts_ms").alias("cdc_ts_ms"),
        F.col("_cdc_lsn").alias("cdc_lsn"),
        F.col("_raw_ingested_at").alias("raw_ingested_at"),
        F.coalesce(F.col("_op") == "d", F.lit(False)).alias("is_deleted"),
        F.lit(ctx.src_version_to).cast("bigint").alias("src_version"),
        F.lit(ctx.run_id).alias("last_run_id"),
        F.current_timestamp().alias("updated_at"),
    )


def state_order_sql(alias: str) -> str:
    """Thứ tự event của dòng state dạng SQL — cùng quy tắc với event_order()/load_state (NULL -> -1 / MIN_TS, del).
    Vd state_order_sql("s") -> "struct(coalesce(s.cdc_ts_ms, -1L), ...)" """
    return (f"struct(coalesce({alias}.cdc_ts_ms, -1L), coalesce({alias}.cdc_lsn, -1L), "
            f"coalesce({alias}.raw_ingested_at, TIMESTAMP'{MIN_TS:%Y-%m-%d %H:%M:%S}'), "
            f"coalesce(cast({alias}.is_deleted AS INT), 0))")


# State không bao giờ lùi: chỉ ghi đè khi event mới ≥ event đang lưu (bằng nhau vẫn ghi để cập nhật last_run_id)
STATE_NOT_OLDER = f"{state_order_sql('s')} >= {state_order_sql('t')}"


def src_partition_cond(src_tbl: Optional[str]) -> str:
    """[03/10 3P] Điều kiện thêm vào MERGE bảng control partition theo src_tbl (ctrl_cdc_state, ctrl_cdc_reject):
    MERGE chỉ đọc partition của nguồn mình -> 2 nguồn ghi song song không xung đột (Delta xét xung đột theo partition).
    Vd src_partition_cond("brz_3rd_crawler_poi_stream") -> " AND t.src_tbl = 'brz_3rd_crawler_poi_stream'" ; None -> "" (như cũ)"""
    if not src_tbl:
        return ""
    if not is_identifier(src_tbl):
        raise ValueError(f"src_tbl không hợp lệ: {src_tbl!r}")
    return f" AND t.src_tbl = '{src_tbl}'"


def commit_state(results: list, src_tbl: Optional[str] = None) -> None:
    """1 MERGE state của các bảng SUCCESS. Bảng lỗi không ghi state -> lần sau xử lý lại toàn bộ event của bảng đó.
    Dòng khớp chỉ cập nhật khi thứ tự event mới ≥ thứ tự đang lưu (STATE_NOT_OLDER): run cũ/chậm commit sau
    không kéo state lùi về event cũ hơn.
    [03/10 3P] src_tbl: thêm điều kiện partition (src_partition_cond); None = như cũ.

    Input : results — list[TableResult] (dùng state_df của bảng SUCCESS) ; src_tbl — nguồn của lần chạy
    Output: không
    """
    parts = [r.state_df for r in results if r.status == "SUCCESS" and r.state_df is not None]
    if not parts:
        return
    source = reduce(DataFrame.unionByName, parts).select(*STATE_COLUMNS)
    cond = "t.trg_schema = s.trg_schema AND t.trg_tbl = s.trg_tbl AND t.entity_key = s.entity_key" + src_partition_cond(src_tbl)
    with_retry(lambda: (
        DeltaTable.forName(spark, T_CDC_STATE).alias("t")
        .merge(source.alias("s"), cond)
        .whenMatchedUpdateAll(condition=STATE_NOT_OLDER)
        .whenNotMatchedInsertAll()
        .execute()
    ), "ctrl_cdc_state")
    log(f"ctrl_cdc_state: MERGE {len(parts)} bảng")


def merge_rejects(parts: list, note: str = "", src_tbl: Optional[str] = None) -> None:
    """[03/10 3P] MERGE event lỗi vào ctrl_cdc_reject theo event_hash (phần dùng chung của commit_rejects partner).
    Event đã có (đọc lại) -> reject_count + 1, cập nhật lần gần nhất/lý do; không sinh dòng trùng.
    src_tbl: thêm điều kiện partition (src_partition_cond); None = chỉ theo event_hash.

    Input : parts — list[DataFrame] đúng cột ctrl_cdc_reject (None bị bỏ qua) ; note — ghi thêm vào log ; src_tbl
    Output: không
    Ví dụ : merge_rejects([reject_df_bang_a, reject_df_phan_loai], "phân loại=3, theo bảng=1")
    """
    parts = [p for p in parts if p is not None]
    if not parts:
        return
    source = reduce(DataFrame.unionByName, parts).dropDuplicates(["event_hash"])
    with_retry(lambda: (
        DeltaTable.forName(spark, T_CDC_REJECT).alias("t")
        .merge(source.alias("s"), "t.event_hash = s.event_hash" + src_partition_cond(src_tbl))
        .whenMatchedUpdate(set={
            "last_run_id": "s.last_run_id",
            "reject_count": "t.reject_count + 1",
            "last_rejected_at": "s.last_rejected_at",
            "reject_reason": "s.reject_reason",
            "reject_detail": "s.reject_detail",
        })
        .whenNotMatchedInsertAll()
        .execute()
    ), "ctrl_cdc_reject")
    log(f"ctrl_cdc_reject: MERGE ({note})" if note else "ctrl_cdc_reject: MERGE")


## 14. table_flow_utils — Khung chạy dùng chung

**[03/10 3P]** Phần điều khiển giống hệt nhau giữa các notebook extract (chuyển từ partner `process_table`, `run_waves`, `run_extract`).
Notebook chỉ còn viết phần riêng: đọc + phân loại raw, xử lý 1 bảng, commit.

| Hàm | Mục đích |
|---|---|
| `process_with_retry(spec, wave, ctx, body)` | Chạy `body(res)` cho 1 bảng; lỗi tạm thời → thử lại (`max_retries`); lỗi khác → FAILED. Không bao giờ raise |
| `run_tables(specs, events_for, worker, ctx, results)` | Bảng 0 event → NO_DATA (không tốn job); bảng còn lại chạy theo wave priority (`run_in_waves`) |
| `run_controlled(ctx, body)` | Cả lần chạy: nhận khoá (không được → `SKIPPED_CONCURRENT`) → run log RUNNING → chốt version raw → `body(out)` → watermark → run log → nhả khoá → giải phóng cache → JSON tóm tắt |

`body(out: RunOutcome)` của notebook phải: đọc + xử lý + commit, điền `out.profile`, `out.results`, `out.cached` và đặt
`out.status = decide_status(out.results)` sau khi xử lý xong. Lỗi bất kỳ → run FAILED, watermark giữ nguyên, vẫn đóng log và nhả khoá.


In [ ]:
# =============================================================================
# 14. table_flow_utils  [03/10 3P: chuyển từ NB_EXTRACT_PARTNER_CDC_BRZ_TO_SLV]
# =============================================================================
def process_with_retry(spec: TableSpec, wave: int, ctx: RunContext, body: Callable) -> TableResult:
    """Chạy body(res) cho 1 bảng trong luồng của wave. Lỗi tạm thời -> thử lại (ctx.max_retries); lỗi khác -> FAILED.
    Không bao giờ raise: bảng lỗi không làm dừng bảng khác. Lỗi thì giải phóng cache, bỏ state/reject chờ commit.

    Input : spec ; wave ; ctx ; body(res: TableResult) -> None — các bước của 1 bảng, ghi số đếm vào res, lỗi thì raise
    Output: TableResult (SUCCESS | FAILED)
    Ví dụ : process_with_retry(spec, 1, ctx, lambda res: _run_table(res, valid_df, state_df, ctx))
    """
    res = TableResult(spec=spec, wave=wave, started_at=utc_now())
    spark.sparkContext.setJobDescription(f"{ctx.nb_name} | w{wave} | {spec.trg_tbl}")
    try:
        for attempt in range(1, ctx.max_retries + 2):
            res.attempt_no = attempt
            try:
                body(res)
                res.status, res.error_message = "SUCCESS", None
                break
            except Exception as e:
                res.release()
                res.state_df = res.reject_df = None
                res.error_message = short_error(e)
                if attempt <= ctx.max_retries and is_transient(e):
                    log(f"lỗi tạm thời, thử lại lần {attempt + 1}: {res.error_message[:300]}", "WARN", wave, spec.trg_tbl)
                    time.sleep(5 * attempt)
                    continue
                res.status = "FAILED"
                log(f"FAILED: {res.error_message[:1000]}", "ERROR", wave, spec.trg_tbl)
                break
    finally:
        res.ended_at = utc_now()
        spark.sparkContext.setJobDescription(None)

    if res.status == "SUCCESS":
        log(f"ok  in={res.input_rows} stale={res.stale_rows} apply={res.applicable_rows} "
            f"ins={res.inserted_rows} upd={res.updated_rows} del={res.delete_event_rows} "
            f"rej={res.rejected_rows} null_cast={res.cast_null_rows}  {res.duration_ms / 1000:.1f}s",
            wave=wave, table=spec.trg_tbl)
    return res


def run_tables(specs: list, events_for: Callable, worker: Callable, ctx: RunContext, results: list) -> None:
    """Bảng không có event -> NO_DATA (không tốn job). Bảng còn lại chạy theo wave priority (run_in_waves);
    stop_on_failure: wave có bảng lỗi thì các wave sau SKIPPED.

    Input : specs ; events_for(spec) -> int: số event/tài liệu thuộc bảng ; worker(spec, wave) -> TableResult ;
            ctx ; results — list nhận kết quả
    Output: không (thêm vào results)
    Ví dụ : run_tables(specs, profile.events_for, lambda s, w: process_table(s, valid_df, state_df, ctx, w), ctx, out)
    """
    waves = {}
    for spec in specs:
        if events_for(spec) == 0:
            now = utc_now()
            results.append(TableResult(spec=spec, status="NO_DATA", started_at=now, ended_at=now))
        else:
            waves.setdefault(spec.priority, []).append(spec)

    def skipped(spec: TableSpec, wave: int) -> TableResult:
        now = utc_now()
        return TableResult(spec=spec, wave=wave, status="SKIPPED", started_at=now, ended_at=now,
                           error_message="Không chạy do wave trước lỗi (stop_on_failure)")

    run_in_waves(
        waves,
        worker=worker,
        max_parallel=ctx.max_parallel,
        results=results,
        is_failed=lambda r: r.status == "FAILED",
        on_skip=skipped,
        stop_on_failure=ctx.stop_on_failure,
        label=lambda spec: spec.trg_tbl,
    )


def run_controlled(ctx: RunContext, body: Callable) -> dict:
    """Luồng điều khiển 1 lần chạy (gọi sau load_table_specs + load_watermark):
    nhận khoá -> run log RUNNING -> chốt version raw (pin_snapshot) -> body(out) -> finalize_watermark -> run log
    -> nhả khoá -> giải phóng cache. dry_run: không khoá, không ghi control. Không nhận được khoá -> SKIPPED_CONCURRENT.
    Lỗi trong body -> FAILED (watermark giữ nguyên), vẫn đóng run log và nhả khoá.

    Input : ctx ; body(out: RunOutcome) -> None — phần riêng của notebook (đọc, xử lý, commit), đặt out.status
    Output: dict JSON tóm tắt (build_summary + out.extra)
    Ví dụ : RUN_SUMMARY = run_controlled(ctx, lambda out: process_run(ctx, specs, out))
    """
    log(f"ctrl={CTRL_SCHEMA}")
    if not ctx.dry_run:
        running = {"status": F.lit("RUNNING"), "last_run_id": F.lit(ctx.run_id)} if ctx.updates_watermark else None
        try:
            acquired, holder = acquire_lock(ctx, running)
        except Exception:
            release_lock(ctx)                     # lỗi sau khi đã ghi khoá: nhả (chỉ nhả nếu đúng exec_id này)
            raise
        if not acquired:
            msg = f"Đang có lần chạy khác: {holder}"
            log(f"{msg} -> bỏ qua", "WARN")
            write_run_log(run_log_row(ctx, "SKIPPED_CONCURRENT", error=msg, ended_at=utc_now()))
            return build_summary(ctx, "SKIPPED_CONCURRENT", None, [], msg)
        log(f"nhận khoá ({ctx.lock_mode})")

    out = RunOutcome()
    t0 = time.time()
    try:
        if not ctx.dry_run:
            write_run_log(run_log_row(ctx, "RUNNING"))
        pin_snapshot(ctx)
        log(f"mở run log, chốt version {ctx.src_version_to} ({time.time() - t0:.0f}s)")
        body(out)
        failed = [r for r in out.results if r.status in ("FAILED", "SKIPPED")]
        out.error = "; ".join(f"{r.spec.trg_tbl}: {(r.error_message or '')[:300]}" for r in failed)[:4000] or None
    except Exception as e:
        out.status, out.error = "FAILED", short_error(e)
        log(traceback.format_exc(), "ERROR")
    finally:
        if not ctx.dry_run:
            # Tách 3 khối: watermark lỗi vẫn phải đóng run log; khoá nhả sau cùng (không để RUNNING/khoá treo)
            try:
                note = finalize_watermark(ctx, out.status, out.profile.watermark_to if out.profile else None, out.error)
                if note:
                    out.error = f"{out.error} | {note}" if out.error else note
            except Exception as e:
                out.status, out.error = "FAILED", f"{out.error or ''} | ghi watermark lỗi: {short_error(e, 1000)}"
                log(traceback.format_exc(), "ERROR")
            try:
                write_run_log(run_log_row(ctx, out.status, out.profile, out.results, out.error, ended_at=utc_now()))
            except Exception as e:
                out.status, out.error = "FAILED", f"{out.error or ''} | ghi run log lỗi: {short_error(e, 1000)}"
                log(traceback.format_exc(), "ERROR")
            try:
                release_lock(ctx)
            except Exception as e:
                out.error = f"{out.error or ''} | nhả khoá lỗi (khoá tự hết hạn sau {ctx.running_timeout_minutes} phút): " \
                            f"{short_error(e, 1000)}"
                log(traceback.format_exc(), "ERROR")
        for r in out.results:
            r.release()
        for df in out.cached:
            if df is not None:
                df.unpersist()

    summary = build_summary(ctx, out.status, out.profile, out.results, out.error)
    summary.update(out.extra)
    return summary
